# 1. Dependencias

In [1]:
# Instalar dependencias necesarias para la evaluación Zero-Day
!pip install -q \
    h5py \
    scikit-learn \
    matplotlib \
    seaborn \
    tqdm \
    PyYAML \
    einops

In [2]:
import torch
import numpy as np
import h5py
import sklearn
import matplotlib
import seaborn
import yaml
import einops
import tqdm

print("PyTorch:", torch.__version__)
print("CUDA disponible:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("No se detectó GPU. Activa una GPU T4 en Colab.")

PyTorch: 2.11.0+cu128
CUDA disponible: True
GPU: Tesla T4


# 2. Cargar .zip al colab

In [3]:
# Celda 1: Montar Google Drive
from google.colab import drive
drive.mount("/content/drive")

# Verificar espacio, integridad y descomprimir en /content
import os
import shutil
import zipfile

ZIP_PATH = "/content/drive/MyDrive/proyecto_ids_vit_colab.zip"
LOCAL_ZIP = "/content/proyecto_ids_vit.zip"
EXTRACT_DIR = "/content/osr_workspace"

if not os.path.isfile(ZIP_PATH):
    raise FileNotFoundError(f"No existe el ZIP: {ZIP_PATH}")

zip_size = os.path.getsize(ZIP_PATH)

with zipfile.ZipFile(ZIP_PATH, "r") as zip_file:
    bad_file = zip_file.testzip()
    if bad_file is not None:
        raise RuntimeError(f"Archivo corrupto dentro del ZIP: {bad_file}")

    members = zip_file.infolist()
    uncompressed_size = sum(member.file_size for member in members)

free_space = shutil.disk_usage("/content").free
required_space = zip_size + uncompressed_size + (2 * 1024**3)

print(f"Tamaño ZIP:           {zip_size / 1024**3:.2f} GB")
print(f"Tamaño descomprimido: {uncompressed_size / 1024**3:.2f} GB")
print(f"Espacio libre:        {free_space / 1024**3:.2f} GB")
print(f"Archivos en ZIP:      {len(members):,}")

if free_space < required_space:
    raise RuntimeError(
        "No existe suficiente espacio en /content para copiar y descomprimir el proyecto."
    )

if os.path.exists(EXTRACT_DIR):
    shutil.rmtree(EXTRACT_DIR)

os.makedirs(EXTRACT_DIR, exist_ok=True)

print("\nCopiando ZIP desde Drive...")
shutil.copy2(ZIP_PATH, LOCAL_ZIP)

print("Descomprimiendo...")
with zipfile.ZipFile(LOCAL_ZIP, "r") as zip_file:
    base = os.path.abspath(EXTRACT_DIR)

    for member in zip_file.infolist():
        destination = os.path.abspath(
            os.path.join(EXTRACT_DIR, member.filename)
        )

        if destination != base and not destination.startswith(base + os.sep):
            raise RuntimeError(
                f"Ruta insegura dentro del ZIP: {member.filename}"
            )

    zip_file.extractall(EXTRACT_DIR)

print("✓ Descompresión completada.")

Mounted at /content/drive
Tamaño ZIP:           2.22 GB
Tamaño descomprimido: 14.09 GB
Espacio libre:        63.34 GB
Archivos en ZIP:      5,603

Copiando ZIP desde Drive...
Descomprimiendo...
✓ Descompresión completada.


# 3. Evaluación de integridad

In [4]:
# Celda 2.1: preparación y localización de manifiestos
import os
import json
import glob
import hashlib
from pathlib import Path

PROJECT_ROOT = "/content/osr_workspace/proyecto_ids_vit_colab"
BACKUP_ROOT = "/content/drive/MyDrive/OSR_ViT_Backup_FULL"

FILES_MANIFEST = os.path.join(BACKUP_ROOT, "migration_metadata/files_sha256.txt")
MANIFESTS_MANIFEST = os.path.join(BACKUP_ROOT, "migration_metadata/manifests_sha256.txt")
INVENTORY_FILE = os.path.join(BACKUP_ROOT, "migration_metadata/files_inventory_v2.tsv")

required_paths = {
    "Raíz local": PROJECT_ROOT,
    "Respaldo Drive": BACKUP_ROOT,
    "Manifiesto de archivos": FILES_MANIFEST,
    "Manifiesto de manifiestos": MANIFESTS_MANIFEST,
    "Inventario": INVENTORY_FILE,
}

for description, path in required_paths.items():
    if description in {"Raíz local", "Respaldo Drive"}:
        if not os.path.isdir(path):
            raise FileNotFoundError(f"{description} no existe: {path}")
    elif not os.path.isfile(path):
        raise FileNotFoundError(f"{description} no existe: {path}")

os.chdir(PROJECT_ROOT)

def sha256_file(path, block_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as file:
        for block in iter(lambda: file.read(block_size), b""):
            digest.update(block)
    return digest.hexdigest()

print("Raíz local del proyecto:", PROJECT_ROOT)
print("Respaldo en Drive:       ", BACKUP_ROOT)
print("Manifiesto principal:    ", FILES_MANIFEST)
print("Manifiesto superior:     ", MANIFESTS_MANIFEST)
print("Inventario:              ", INVENTORY_FILE)

print("\nPrimeras líneas de files_sha256.txt:")
with open(FILES_MANIFEST, "r", encoding="utf-8") as file:
    for _ in range(5):
        line = file.readline()
        if not line:
            break
        print(line.rstrip())

print("\n✓ Rutas y archivos de integridad localizados correctamente.")

FileNotFoundError: Respaldo Drive no existe: /content/drive/MyDrive/OSR_ViT_Backup_FULL

In [ ]:
# Celda 2.2 — Verificación SHA-256 de data/processed sin detener la ejecución

import os
import re
import json
from datetime import datetime, timezone
from collections import Counter, defaultdict
from tqdm.auto import tqdm

SPLITS = ("train_known", "val_known", "test_known", "test_ood")
PROCESSED_ROOT = os.path.join(PROJECT_ROOT, "data", "processed")
REPORT_DIR = os.path.join(PROJECT_ROOT, "results")
REPORT_PATH = os.path.join(REPORT_DIR, "data_processed_integrity_report.json")

os.makedirs(REPORT_DIR, exist_ok=True)

report = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "project_root": PROJECT_ROOT,
    "manifest_path": FILES_MANIFEST,
    "status": "UNKNOWN",
    "summary": {},
    "splits": {},
    "missing_files": [],
    "unexpected_files": [],
    "hash_mismatches": [],
    "read_errors": [],
    "manifest_errors": [],
}

# ------------------------------------------------------------
# 1. Leer el manifiesto SHA-256
# ------------------------------------------------------------
expected_files = {}

try:
    with open(FILES_MANIFEST, "r", encoding="utf-8", errors="replace") as file:
        for line_number, raw_line in enumerate(file, start=1):
            line = raw_line.strip()

            if not line or line.startswith("#"):
                continue

            match = re.match(r"^([0-9a-fA-F]{64})\s+[* ]?(.+)$", line)

            if not match:
                report["manifest_errors"].append({
                    "line": line_number,
                    "content": line[:300],
                    "error": "Formato SHA-256 no reconocido",
                })
                continue

            expected_sha = match.group(1).lower()
            relative_path = match.group(2).strip().replace("\\", "/")

            while relative_path.startswith("./"):
                relative_path = relative_path[2:]

            if not relative_path.startswith("data/processed/"):
                continue

            parts = relative_path.split("/")

            if len(parts) < 4 or parts[2] not in SPLITS:
                continue

            if not relative_path.endswith(".hdf5"):
                continue

            if relative_path in expected_files:
                report["manifest_errors"].append({
                    "line": line_number,
                    "content": relative_path,
                    "error": "Ruta duplicada en el manifiesto",
                })
                continue

            expected_files[relative_path] = expected_sha

except Exception as error:
    report["manifest_errors"].append({
        "error": f"No se pudo leer el manifiesto: {type(error).__name__}: {error}"
    })

# ------------------------------------------------------------
# 2. Inventariar archivos HDF5 locales
# ------------------------------------------------------------
actual_files = {}

for split in SPLITS:
    split_dir = os.path.join(PROCESSED_ROOT, split)

    if not os.path.isdir(split_dir):
        report["read_errors"].append({
            "path": split_dir,
            "error": "Directorio ausente",
        })
        continue

    try:
        for root, _, filenames in os.walk(split_dir):
            for filename in filenames:
                if not filename.endswith(".hdf5"):
                    continue

                absolute_path = os.path.join(root, filename)
                relative_path = os.path.relpath(
                    absolute_path,
                    PROJECT_ROOT,
                ).replace("\\", "/")

                actual_files[relative_path] = absolute_path

    except Exception as error:
        report["read_errors"].append({
            "path": split_dir,
            "error": f"{type(error).__name__}: {error}",
        })

# ------------------------------------------------------------
# 3. Detectar faltantes y adicionales
# ------------------------------------------------------------
expected_paths = set(expected_files)
actual_paths = set(actual_files)

missing_paths = sorted(expected_paths - actual_paths)
unexpected_paths = sorted(actual_paths - expected_paths)

report["missing_files"] = missing_paths
report["unexpected_files"] = unexpected_paths

# ------------------------------------------------------------
# 4. Calcular SHA-256 de archivos presentes y esperados
# ------------------------------------------------------------
common_paths = sorted(expected_paths & actual_paths)

for relative_path in tqdm(
    common_paths,
    desc="Verificando SHA-256",
    unit="archivo",
):
    absolute_path = actual_files[relative_path]

    try:
        actual_sha = sha256_file(absolute_path)
        expected_sha = expected_files[relative_path]

        if actual_sha.lower() != expected_sha.lower():
            report["hash_mismatches"].append({
                "path": relative_path,
                "expected_sha256": expected_sha,
                "actual_sha256": actual_sha,
                "size_bytes": os.path.getsize(absolute_path),
            })

    except Exception as error:
        report["read_errors"].append({
            "path": relative_path,
            "error": f"{type(error).__name__}: {error}",
        })

# ------------------------------------------------------------
# 5. Resumen por split
# ------------------------------------------------------------
expected_by_split = Counter()
actual_by_split = Counter()
missing_by_split = Counter()
unexpected_by_split = Counter()
mismatch_by_split = Counter()
errors_by_split = Counter()

def split_from_path(path):
    parts = path.replace("\\", "/").split("/")
    if len(parts) >= 3 and parts[2] in SPLITS:
        return parts[2]
    return "unknown"

for path in expected_paths:
    expected_by_split[split_from_path(path)] += 1

for path in actual_paths:
    actual_by_split[split_from_path(path)] += 1

for path in missing_paths:
    missing_by_split[split_from_path(path)] += 1

for path in unexpected_paths:
    unexpected_by_split[split_from_path(path)] += 1

for item in report["hash_mismatches"]:
    mismatch_by_split[split_from_path(item["path"])] += 1

for item in report["read_errors"]:
    errors_by_split[split_from_path(item.get("path", ""))] += 1

for split in SPLITS:
    report["splits"][split] = {
        "expected": expected_by_split[split],
        "actual": actual_by_split[split],
        "missing": missing_by_split[split],
        "unexpected": unexpected_by_split[split],
        "hash_mismatches": mismatch_by_split[split],
        "read_errors": errors_by_split[split],
        "integrity_ok": (
            missing_by_split[split] == 0
            and unexpected_by_split[split] == 0
            and mismatch_by_split[split] == 0
            and errors_by_split[split] == 0
            and expected_by_split[split] > 0
        ),
    }

# ------------------------------------------------------------
# 6. Estado global y persistencia
# ------------------------------------------------------------
integrity_ok = (
    len(expected_files) > 0
    and not report["missing_files"]
    and not report["unexpected_files"]
    and not report["hash_mismatches"]
    and not report["read_errors"]
    and not report["manifest_errors"]
)

report["status"] = "PASS" if integrity_ok else "FAIL"

report["summary"] = {
    "expected_files": len(expected_files),
    "actual_files": len(actual_files),
    "verified_files": len(common_paths),
    "missing_files": len(report["missing_files"]),
    "unexpected_files": len(report["unexpected_files"]),
    "hash_mismatches": len(report["hash_mismatches"]),
    "read_errors": len(report["read_errors"]),
    "manifest_errors": len(report["manifest_errors"]),
}

try:
    temporary_report = REPORT_PATH + ".tmp"

    with open(temporary_report, "w", encoding="utf-8") as file:
        json.dump(
            report,
            file,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
        )

    os.replace(temporary_report, REPORT_PATH)

except Exception as error:
    print(
        f"[!] No se pudo guardar el reporte: "
        f"{type(error).__name__}: {error}"
    )

# ------------------------------------------------------------
# 7. Mostrar resultados sin detener el notebook
# ------------------------------------------------------------
print("\n" + "=" * 86)
print("VERIFICACIÓN DE INTEGRIDAD — data/processed")
print("=" * 86)

print(
    f"{'Split':<15}"
    f"{'Esperados':>12}"
    f"{'Locales':>12}"
    f"{'Faltantes':>12}"
    f"{'Extras':>10}"
    f"{'SHA incorrecto':>16}"
    f"{'Estado':>10}"
)

print("-" * 86)

for split in SPLITS:
    values = report["splits"][split]

    print(
        f"{split:<15}"
        f"{values['expected']:>12,}"
        f"{values['actual']:>12,}"
        f"{values['missing']:>12,}"
        f"{values['unexpected']:>10,}"
        f"{values['hash_mismatches']:>16,}"
        f"{('OK' if values['integrity_ok'] else 'ERROR'):>10}"
    )

print("-" * 86)
print("Estado global:", report["status"])
print("Reporte:", REPORT_PATH)

if report["missing_files"]:
    print("\nPrimeros archivos faltantes:")
    for path in report["missing_files"][:10]:
        print(" -", path)

if report["unexpected_files"]:
    print("\nPrimeros archivos adicionales:")
    for path in report["unexpected_files"][:10]:
        print(" -", path)

if report["hash_mismatches"]:
    print("\nPrimeros archivos con SHA-256 incorrecto:")
    for item in report["hash_mismatches"][:10]:
        print(" -", item["path"])

if report["read_errors"]:
    print("\nPrimeros errores de lectura:")
    for item in report["read_errors"][:10]:
        print(" -", item)

if report["manifest_errors"]:
    print("\nPrimeros errores del manifiesto:")
    for item in report["manifest_errors"][:10]:
        print(" -", item)

if integrity_ok:
    print("\n✓ Integridad completa. Se puede continuar con la evaluación.")
else:
    print(
        "\n⚠ Se encontraron discrepancias. "
        "La celda terminó sin interrumpir el notebook, "
        "pero no debe iniciarse la evaluación hasta revisarlas."
    )

DATA_INTEGRITY_OK = integrity_ok

Verificando SHA-256:   0%|          | 0/5571 [00:00<?, ?archivo/s]


VERIFICACIÓN DE INTEGRIDAD — data/processed
Split             Esperados     Locales   Faltantes    Extras  SHA incorrecto    Estado
--------------------------------------------------------------------------------------
train_known           1,852       1,852           0         0               0        OK
val_known             1,849       1,849           0         0               0        OK
test_known            1,840       1,840           0         0               0        OK
test_ood                 30          30           0         0               0        OK
--------------------------------------------------------------------------------------
Estado global: PASS
Reporte: /content/osr_workspace/proyecto_ids_vit_colab/results/data_processed_integrity_report.json

✓ Integridad completa. Se puede continuar con la evaluación.


In [ ]:
# Celda 2.3 — Validación metodológica de train_known contra scaler_bounds.json

import os
import json
from datetime import datetime, timezone

SCALER_JSON = os.path.join(PROJECT_ROOT, "configs", "scaler_bounds.json")
TRAIN_DIR = os.path.join(PROJECT_ROOT, "data", "processed", "train_known")
REPORT_PATH = os.path.join(
    PROJECT_ROOT,
    "results",
    "train_known_scaler_integrity_report.json",
)

report = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "status": "UNKNOWN",
    "scaler_path": SCALER_JSON,
    "train_dir": TRAIN_DIR,
    "checks": {},
    "missing_files": [],
    "unexpected_files": [],
    "size_mismatches": [],
    "errors": [],
}

try:
    with open(SCALER_JSON, "r", encoding="utf-8") as file:
        scaler = json.load(file)
except Exception as error:
    scaler = {}
    report["errors"].append(
        f"No se pudo leer scaler_bounds.json: {type(error).__name__}: {error}"
    )

metadata = scaler.get("metadata", {})
manifest_files = scaler.get("manifest", {}).get("files", [])

report["checks"]["source_split"] = {
    "expected": "train_known",
    "actual": metadata.get("source_split"),
    "ok": metadata.get("source_split") == "train_known",
}

report["checks"]["taxonomy"] = {
    "expected": ["Benign", "DoS", "DDoS"],
    "actual": metadata.get("taxonomy"),
    "ok": metadata.get("taxonomy") == ["Benign", "DoS", "DDoS"],
}

report["checks"]["tensor_shape"] = {
    "expected": [18, 144, 3],
    "actual": metadata.get("tensor_shape"),
    "ok": metadata.get("tensor_shape") == [18, 144, 3],
}

report["checks"]["source_directory"] = {
    "expected_suffix": "data/processed/train_known/",
    "actual": metadata.get("source_directory"),
    "ok": str(metadata.get("source_directory", "")).replace("\\", "/").endswith(
        "data/processed/train_known/"
    ),
}

expected_count_metadata = int(metadata.get("hdf5_file_count", -1))
expected_count_manifest = len(manifest_files)

actual_files = {}

if os.path.isdir(TRAIN_DIR):
    for filename in os.listdir(TRAIN_DIR):
        if filename.endswith(".hdf5"):
            path = os.path.join(TRAIN_DIR, filename)
            try:
                actual_files[filename] = os.path.getsize(path)
            except Exception as error:
                report["errors"].append(
                    f"No se pudo leer tamaño de {filename}: "
                    f"{type(error).__name__}: {error}"
                )
else:
    report["errors"].append(f"Directorio ausente: {TRAIN_DIR}")

expected_files = {}

for item in manifest_files:
    try:
        relative_path = str(item["relative_path"])
        size_bytes = int(item["size_bytes"])
        expected_files[relative_path] = size_bytes
    except Exception as error:
        report["errors"].append(
            f"Entrada inválida en manifest del scaler: {item} | "
            f"{type(error).__name__}: {error}"
        )

expected_names = set(expected_files)
actual_names = set(actual_files)

report["missing_files"] = sorted(expected_names - actual_names)
report["unexpected_files"] = sorted(actual_names - expected_names)

for filename in sorted(expected_names & actual_names):
    expected_size = expected_files[filename]
    actual_size = actual_files[filename]

    if expected_size != actual_size:
        report["size_mismatches"].append({
            "file": filename,
            "expected_size_bytes": expected_size,
            "actual_size_bytes": actual_size,
        })

report["checks"]["hdf5_count_metadata"] = {
    "expected": expected_count_metadata,
    "actual": len(actual_files),
    "ok": expected_count_metadata == len(actual_files),
}

report["checks"]["hdf5_count_manifest"] = {
    "expected": expected_count_manifest,
    "actual": len(actual_files),
    "ok": expected_count_manifest == len(actual_files),
}

report["checks"]["manifest_internal_consistency"] = {
    "metadata_count": expected_count_metadata,
    "manifest_count": expected_count_manifest,
    "ok": expected_count_metadata == expected_count_manifest,
}

all_checks_ok = all(
    check.get("ok", False)
    for check in report["checks"].values()
)

integrity_ok = (
    all_checks_ok
    and not report["missing_files"]
    and not report["unexpected_files"]
    and not report["size_mismatches"]
    and not report["errors"]
)

report["status"] = "PASS" if integrity_ok else "FAIL"

try:
    os.makedirs(os.path.dirname(REPORT_PATH), exist_ok=True)

    temporary_path = REPORT_PATH + ".tmp"

    with open(temporary_path, "w", encoding="utf-8") as file:
        json.dump(
            report,
            file,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
        )

    os.replace(temporary_path, REPORT_PATH)

except Exception as error:
    print(
        f"[!] No se pudo guardar el reporte: "
        f"{type(error).__name__}: {error}"
    )

print("\n" + "=" * 72)
print("VALIDACIÓN train_known CONTRA scaler_bounds.json")
print("=" * 72)

for name, check in report["checks"].items():
    print(f"{name:<32}: {'OK' if check['ok'] else 'ERROR'}")

print("-" * 72)
print("Archivos esperados por metadata :", expected_count_metadata)
print("Archivos esperados por manifest :", expected_count_manifest)
print("Archivos locales                :", len(actual_files))
print("Faltantes                       :", len(report["missing_files"]))
print("Adicionales                     :", len(report["unexpected_files"]))
print("Tamaños incorrectos             :", len(report["size_mismatches"]))
print("Errores                         :", len(report["errors"]))
print("Estado global                   :", report["status"])
print("Reporte                         :", REPORT_PATH)

if report["missing_files"]:
    print("\nPrimeros faltantes:")
    for path in report["missing_files"][:10]:
        print(" -", path)

if report["unexpected_files"]:
    print("\nPrimeros adicionales:")
    for path in report["unexpected_files"][:10]:
        print(" -", path)

if report["size_mismatches"]:
    print("\nPrimeros tamaños incorrectos:")
    for item in report["size_mismatches"][:10]:
        print(" -", item)

if report["errors"]:
    print("\nPrimeros errores:")
    for error in report["errors"][:10]:
        print(" -", error)

if integrity_ok:
    print("\n✓ train_known coincide con el scaler usado originalmente.")
else:
    print(
        "\n⚠ Se detectaron discrepancias. "
        "No debe iniciarse la evaluación hasta revisarlas."
    )

SCALER_INTEGRITY_OK = integrity_ok


VALIDACIÓN train_known CONTRA scaler_bounds.json
source_split                    : OK
taxonomy                        : OK
tensor_shape                    : OK
source_directory                : OK
hdf5_count_metadata             : OK
hdf5_count_manifest             : OK
manifest_internal_consistency   : OK
------------------------------------------------------------------------
Archivos esperados por metadata : 1852
Archivos esperados por manifest : 1852
Archivos locales                : 1852
Faltantes                       : 0
Adicionales                     : 0
Tamaños incorrectos             : 0
Errores                         : 0
Estado global                   : PASS
Reporte                         : /content/osr_workspace/proyecto_ids_vit_colab/results/train_known_scaler_integrity_report.json

✓ train_known coincide con el scaler usado originalmente.


# 4. Validaciones Adicionales

In [5]:
# Celda 4.1 — Parchear torch.load en evaluate_zero_day.py sin tocar vit_ablation.py

import os
import re

EVALUATOR_PATH = os.path.join(PROJECT_ROOT, "src", "evaluation", "evaluate_zero_day.py")
VIT_PATH = os.path.join(PROJECT_ROOT, "src", "models", "vit_ablation.py")

with open(EVALUATOR_PATH, "r", encoding="utf-8") as file:
    code = file.read()

original_code = code

# Agrega weights_only=False a llamadas torch.load(...) que aún no lo tienen
pattern = r"torch\.load\(([^)]*?)\)"

def add_weights_only_false(match):
    call_args = match.group(1)

    if "weights_only" in call_args:
        return match.group(0)

    return f"torch.load({call_args}, weights_only=False)"

code = re.sub(pattern, add_weights_only_false, code)

if code == original_code:
    print("[!] No se modificó evaluate_zero_day.py. Revisa si ya estaba parcheado o si usa torch.load multilinea compleja.")
else:
    with open(EVALUATOR_PATH, "w", encoding="utf-8") as file:
        file.write(code)

    print("✓ Parche aplicado en evaluate_zero_day.py")

# Confirmar que vit_ablation.py no fue tocado
with open(VIT_PATH, "r", encoding="utf-8") as file:
    vit_code = file.read()

print("evaluate_zero_day.py tiene weights_only=False:", "weights_only=False" in code)
print("vit_ablation.py tiene weights_only=False:", "weights_only=False" in vit_code)

✓ Parche aplicado en evaluate_zero_day.py
evaluate_zero_day.py tiene weights_only=False: True
vit_ablation.py tiene weights_only=False: False


In [6]:
# Celda 4.2 — Configurar OSR base para PCA=0.999999 y barrido de lambdas

import os
import yaml

CONFIG_PATH = os.path.join(PROJECT_ROOT, "configs", "global_config.yaml")

with open(CONFIG_PATH, "r", encoding="utf-8") as file:
    config = yaml.safe_load(file)

config["osr_shield"]["pca_variance_retained"] = 0.999999
config["osr_shield"]["lambda_candidates"] = [3.0, 3.5, 4.0, 4.5, 5.0, 6.0, 7.0]
config["osr_shield"]["selected_lambda_mad"] = None
config["osr_shield"]["max_val_known_frr"] = 0.10
config["osr_shield"]["ridge_epsilon_init"] = 1.0e-5
config["osr_shield"]["quantile_count"] = 10000
config["osr_shield"]["quantile_subsample_size"] = 100000

with open(CONFIG_PATH, "w", encoding="utf-8") as file:
    yaml.safe_dump(
        config,
        file,
        sort_keys=False,
        allow_unicode=True,
    )

print("✓ global_config.yaml actualizado para la nueva estrategia OSR")
print("pca_variance_retained:", config["osr_shield"]["pca_variance_retained"])
print("lambda_candidates:", config["osr_shield"]["lambda_candidates"])
print("selected_lambda_mad:", config["osr_shield"]["selected_lambda_mad"])

✓ global_config.yaml actualizado para la nueva estrategia OSR
pca_variance_retained: 0.999999
lambda_candidates: [3.0, 3.5, 4.0, 4.5, 5.0, 6.0, 7.0]
selected_lambda_mad: None


In [31]:
# Celda 4.3 — Parche mínimo PCA variance + validación OSR

import os
import re
import math
import yaml
import shutil
from datetime import datetime

CONFIG_PATH = os.path.join(PROJECT_ROOT, "configs", "global_config.yaml")
EVALUATOR_PATH = os.path.join(PROJECT_ROOT, "src", "evaluation", "evaluate_zero_day.py")

EXPECTED_PCA_VARIANCE = 0.999999
EXPECTED_SELECTED_LAMBDA = None
EXPECTED_LAMBDAS = [1.0, 1.25, 1.5, 1.75, 2.0]
EXPECTED_MAX_VAL_FRR = 0.10

# ------------------------------------------------------------
# 1. Parche mínimo: permitir pca_variance_retained distinto de 0.99
# ------------------------------------------------------------

with open(EVALUATOR_PATH, "r", encoding="utf-8") as file:
    evaluator_code = file.read()

backup_path = (
    EVALUATOR_PATH
    + ".backup_before_pca_variance_patch_"
    + datetime.utcnow().strftime("%Y%m%d_%H%M%S")
)

patched_code = evaluator_code

pattern = re.compile(
    r'''(\s*)pca_variance\s*=\s*float\(osr_config\.get\("pca_variance_retained",\s*0\.99\)\)\n'''
    r'''\1if\s+not\s+math\.isclose\(pca_variance,\s*0\.99,\s*rel_tol=0\.0,\s*abs_tol=1e-12\):\n'''
    r'''\1\s+raise\s+RuntimeError\(f"La metodología fija pca_variance_retained=0\.99; valor configurado=\{pca_variance\}"\)'''
)

replacement = (
    r'''\1pca_variance = float(osr_config.get("pca_variance_retained", 0.99))'''
    "\n"
    r'''\1if not 0.0 < pca_variance <= 1.0:'''
    "\n"
    r'''\1    raise RuntimeError(f"pca_variance_retained debe estar en (0, 1]; valor configurado={pca_variance}")'''
)

patched_code, replacements = pattern.subn(replacement, patched_code)

if replacements > 0:
    shutil.copy2(EVALUATOR_PATH, backup_path)

    with open(EVALUATOR_PATH, "w", encoding="utf-8") as file:
        file.write(patched_code)

    print("✓ Parche mínimo aplicado en evaluate_zero_day.py")
    print("Backup:", backup_path)

elif "La metodología fija pca_variance_retained=0.99" not in evaluator_code:
    print("✓ El evaluador ya no bloquea pca_variance_retained distinto de 0.99")

else:
    print("[ERROR] No se pudo aplicar el parche mínimo automáticamente.")
    print("        Revisa manualmente _resolve_osr_settings() en evaluate_zero_day.py")

# ------------------------------------------------------------
# 2. Validar global_config.yaml
# ------------------------------------------------------------

CONFIG_OK = False

try:
    with open(CONFIG_PATH, "r", encoding="utf-8") as file:
        config = yaml.safe_load(file)

    osr = config.get("osr_shield", {})

    pca_variance = float(osr.get("pca_variance_retained"))
    selected_lambda = osr.get("selected_lambda_mad")
    lambda_candidates = [float(x) for x in osr.get("lambda_candidates", [])]
    max_val_known_frr = float(osr.get("max_val_known_frr", -1))

    checks = {
        "pca_variance_retained == 0.999999": math.isclose(
            pca_variance,
            EXPECTED_PCA_VARIANCE,
            rel_tol=0.0,
            abs_tol=1e-12,
        ),
        "selected_lambda_mad == null": selected_lambda is None,
        "lambda_candidates incluye barrido esperado": all(
            value in lambda_candidates
            for value in EXPECTED_LAMBDAS
        ),
        "max_val_known_frr == 0.10": math.isclose(
            max_val_known_frr,
            EXPECTED_MAX_VAL_FRR,
            rel_tol=0.0,
            abs_tol=1e-12,
        ),
        "ridge_epsilon_init existe": "ridge_epsilon_init" in osr,
        "quantile_count existe": "quantile_count" in osr,
        "quantile_subsample_size existe": "quantile_subsample_size" in osr,
    }

    print("\nCONFIG_PATH:", CONFIG_PATH)
    print("pca_variance_retained:", pca_variance)
    print("selected_lambda_mad:", selected_lambda)
    print("lambda_candidates:", lambda_candidates)
    print("max_val_known_frr:", max_val_known_frr)

    print("\nValidaciones:")
    for name, ok in checks.items():
        print(f" - {name}: {'OK' if ok else 'ERROR'}")

    CONFIG_OK = all(checks.values())

    print("\nEstado global:", "OK" if CONFIG_OK else "ERROR")

except Exception as error:
    print(f"[ERROR] No se pudo validar global_config.yaml: {type(error).__name__}: {error}")
    CONFIG_OK = False

✓ El evaluador ya no bloquea pca_variance_retained distinto de 0.99

CONFIG_PATH: /content/osr_workspace/proyecto_ids_vit_colab/configs/global_config.yaml
pca_variance_retained: 0.999999
selected_lambda_mad: None
lambda_candidates: [1.0, 1.25, 1.5, 1.75, 2.0]
max_val_known_frr: 0.1

Validaciones:
 - pca_variance_retained == 0.999999: OK
 - selected_lambda_mad == null: OK
 - lambda_candidates incluye barrido esperado: OK
 - max_val_known_frr == 0.10: OK
 - ridge_epsilon_init existe: OK
 - quantile_count existe: OK
 - quantile_subsample_size existe: OK

Estado global: OK


/tmp/ipykernel_4538/1338082847.py:28: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  + datetime.utcnow().strftime("%Y%m%d_%H%M%S")


In [32]:
# Celda 4.4 — Compilar evaluate_zero_day.py

import os
import py_compile

EVALUATOR_PATH = os.path.join(PROJECT_ROOT, "src", "evaluation", "evaluate_zero_day.py")

EVALUATOR_COMPILES_OK = False

try:
    py_compile.compile(EVALUATOR_PATH, doraise=True)
    EVALUATOR_COMPILES_OK = True
    print("✓ evaluate_zero_day.py compila correctamente.")

except Exception as error:
    print(f"[ERROR] evaluate_zero_day.py no compila: {type(error).__name__}: {error}")

✓ evaluate_zero_day.py compila correctamente.


In [9]:
# Celda 4.5 — Validar hash de vit_ablation.py contra el checkpoint

import os
import torch

CHECKPOINT_PATH = os.path.join(PROJECT_ROOT, "checkpoints", "vit_nmin_7_checkpoint.pt")
VIT_PATH = os.path.join(PROJECT_ROOT, "src", "models", "vit_ablation.py")

VIT_ORIGINAL_OK = False

def find_key_recursive(obj, target_key):
    if isinstance(obj, dict):
        for key, value in obj.items():
            if key == target_key:
                return value

            found = find_key_recursive(value, target_key)
            if found is not None:
                return found

    elif isinstance(obj, (list, tuple)):
        for item in obj:
            found = find_key_recursive(item, target_key)
            if found is not None:
                return found

    return None

try:
    checkpoint = torch.load(
        CHECKPOINT_PATH,
        map_location="cpu",
        weights_only=False,
    )

    expected_script_sha = find_key_recursive(checkpoint, "script_sha256")
    actual_script_sha = sha256_file(VIT_PATH)

    print("SHA esperado en checkpoint:", expected_script_sha)
    print("SHA actual vit_ablation.py:", actual_script_sha)

    VIT_ORIGINAL_OK = (
        expected_script_sha is not None
        and expected_script_sha == actual_script_sha
    )

    print("Estado:", "OK" if VIT_ORIGINAL_OK else "ERROR")

except Exception as error:
    print(f"[ERROR] No se pudo validar vit_ablation.py: {type(error).__name__}: {error}")

[ERROR] No se pudo validar vit_ablation.py: NameError: name 'sha256_file' is not defined


# 5. Prueba rápida de reproducción

In [12]:
# Reducir num_workers en global_config.yaml para evitar OOM/freeze en Colab

import os
import yaml
import shutil
from datetime import datetime, timezone

PROJECT_ROOT = "/content/osr_workspace/proyecto_ids_vit_colab"
CONFIG_PATH = os.path.join(PROJECT_ROOT, "configs", "global_config.yaml")

NEW_NUM_WORKERS = 2

backup_path = (
    CONFIG_PATH
    + ".backup_num_workers_"
    + datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
)

shutil.copy2(CONFIG_PATH, backup_path)

with open(CONFIG_PATH, "r", encoding="utf-8") as file:
    config = yaml.safe_load(file)

modified_paths = []

def update_num_workers(obj, path="config"):
    if isinstance(obj, dict):
        for key, value in obj.items():
            current_path = f"{path}.{key}"

            if key == "num_workers":
                obj[key] = NEW_NUM_WORKERS
                modified_paths.append((current_path, value, NEW_NUM_WORKERS))
            else:
                update_num_workers(value, current_path)

    elif isinstance(obj, list):
        for index, item in enumerate(obj):
            update_num_workers(item, f"{path}[{index}]")

update_num_workers(config)

with open(CONFIG_PATH, "w", encoding="utf-8") as file:
    yaml.safe_dump(config, file, sort_keys=False, allow_unicode=True)

print("Backup creado:", backup_path)

if modified_paths:
    print("✓ num_workers actualizado:")
    for path, old_value, new_value in modified_paths:
        print(f" - {path}: {old_value} -> {new_value}")
else:
    print("[!] No se encontró ninguna clave num_workers en global_config.yaml")

Backup creado: /content/osr_workspace/proyecto_ids_vit_colab/configs/global_config.yaml.backup_num_workers_20260618_185425
✓ num_workers actualizado:
 - config.data_loader.num_workers: 12 -> 2


In [23]:
# Sección 5 — Prueba rápida de reproducción OSR
# PCA fijo 0.999999 + lambda_mad 3.0

import os
import sys
import json
import yaml
import shutil
import subprocess
from datetime import datetime, timezone

# ------------------------------------------------------------
# 1. Rutas base
# ------------------------------------------------------------

PROJECT_ROOT = "/content/osr_workspace/proyecto_ids_vit_colab"
RESULTS_DIR = os.path.join(PROJECT_ROOT, "results")
CONFIG_PATH = os.path.join(PROJECT_ROOT, "configs", "global_config.yaml")

EVALUATOR_MODULE = "src.evaluation.evaluate_zero_day"
N_MIN = 7

PCA_VARIANCE_REPRO = 0.999999
LAMBDA_REPRO = 3.0

os.makedirs(RESULTS_DIR, exist_ok=True)

REPRO_DIR = os.path.join(
    RESULTS_DIR,
    "reproduction_pca_0999999_lambda_3_0_"
    + datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
)

os.makedirs(REPRO_DIR, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RESULTS_DIR:", RESULTS_DIR)
print("REPRO_DIR:", REPRO_DIR)

# ------------------------------------------------------------
# 2. Validaciones mínimas de existencia
# ------------------------------------------------------------

required_paths = {
    "global_config.yaml": CONFIG_PATH,
    "evaluate_zero_day.py": os.path.join(
        PROJECT_ROOT,
        "src",
        "evaluation",
        "evaluate_zero_day.py",
    ),
    "checkpoint": os.path.join(
        PROJECT_ROOT,
        "checkpoints",
        "vit_nmin_7_checkpoint.pt",
    ),
    "data/processed": os.path.join(
        PROJECT_ROOT,
        "data",
        "processed",
    ),
}

missing_required = []

for name, path in required_paths.items():
    if not os.path.exists(path):
        missing_required.append((name, path))

if missing_required:
    print("[ERROR] Faltan rutas críticas:")
    for name, path in missing_required:
        print(f" - {name}: {path}")

    raise FileNotFoundError(
        "No se puede ejecutar la reproducción porque faltan archivos críticos."
    )

print("✓ Rutas críticas localizadas.")

# ------------------------------------------------------------
# 3. Configurar OSR para la prueba rápida
# ------------------------------------------------------------

with open(CONFIG_PATH, "r", encoding="utf-8") as file:
    config = yaml.safe_load(file)

if "osr_shield" not in config:
    raise KeyError("global_config.yaml no contiene la sección osr_shield")

config["osr_shield"]["pca_variance_retained"] = float(PCA_VARIANCE_REPRO)
config["osr_shield"]["selected_lambda_mad"] = float(LAMBDA_REPRO)

# Mantener candidatos amplios para trazabilidad
config["osr_shield"]["lambda_candidates"] = [
    1.0,
    1.5,
    2.0,
    2.5,
    3.0,
    3.5,
    4.0,
    4.5,
    5.0,
    6.0,
    7.0,
]

config["osr_shield"]["max_val_known_frr"] = 0.10
config["osr_shield"]["ridge_epsilon_init"] = 1.0e-5
config["osr_shield"]["quantile_count"] = 10000
config["osr_shield"]["quantile_subsample_size"] = 100000

with open(CONFIG_PATH, "w", encoding="utf-8") as file:
    yaml.safe_dump(
        config,
        file,
        sort_keys=False,
        allow_unicode=True,
    )

shutil.copy2(
    CONFIG_PATH,
    os.path.join(REPRO_DIR, "global_config_used.yaml"),
)

print("✓ Configuración OSR escrita:")
print("  pca_variance_retained:", config["osr_shield"]["pca_variance_retained"])
print("  selected_lambda_mad:", config["osr_shield"]["selected_lambda_mad"])
print("  lambda_candidates:", config["osr_shield"]["lambda_candidates"])

# ------------------------------------------------------------
# 4. Limpiar salidas previas sin borrar embeddings
# ------------------------------------------------------------

generated_outputs = [
    "open_set_nmin_7.json",
    "open_set_nmin_7.txt",
    "open_set_confusion_nmin_7.png",
    "open_set_roc_nmin_7.png",
    "osr_profiles_nmin_7.pt",
]

removed = []

for filename in generated_outputs:
    path = os.path.join(RESULTS_DIR, filename)

    if os.path.exists(path):
        os.remove(path)
        removed.append(path)

print(f"✓ Archivos previos limpiados: {len(removed)}")
for path in removed:
    print(" -", path)

print("✓ No se borró mlruns/osr_embedding_cache")

# ------------------------------------------------------------
# 5. Ejecutar evaluación Zero-Day
# ------------------------------------------------------------

cmd = [
    sys.executable,
    "-m",
    EVALUATOR_MODULE,
    "--mode",
    "prod",
    "--n_min",
    str(N_MIN),
]

print("\nEjecutando:")
print(" ".join(cmd))
print()

process = subprocess.Popen(
    cmd,
    cwd=PROJECT_ROOT,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

stdout_lines = []

for line in process.stdout:
    print(line, end="")
    stdout_lines.append(line)

returncode = process.wait()
stdout_text = "".join(stdout_lines)

log_path = os.path.join(REPRO_DIR, "execution_log.txt")

with open(log_path, "w", encoding="utf-8") as file:
    file.write(stdout_text)

print("\nReturn code:", returncode)
print("Log guardado en:", log_path)

# ------------------------------------------------------------
# 6. Guardar artefactos generados
# ------------------------------------------------------------

copied_outputs = []

for filename in generated_outputs:
    source = os.path.join(RESULTS_DIR, filename)

    if os.path.exists(source):
        target = os.path.join(REPRO_DIR, filename)
        shutil.copy2(source, target)
        copied_outputs.append(target)

print(f"✓ Artefactos copiados: {len(copied_outputs)}")
for path in copied_outputs:
    print(" -", path)

# ------------------------------------------------------------
# 7. Leer métricas y validar reproducción
# ------------------------------------------------------------

result_json = os.path.join(REPRO_DIR, "open_set_nmin_7.json")

reproduction_summary = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "pca_variance_requested": PCA_VARIANCE_REPRO,
    "lambda_mad": LAMBDA_REPRO,
    "returncode": returncode,
    "status": "UNKNOWN",
    "result_json": result_json,
}

if returncode != 0:
    reproduction_summary["status"] = "ERROR_RETURN_CODE"
    print("\n[ERROR] La evaluación terminó con error. Revisa execution_log.txt.")

elif not os.path.isfile(result_json):
    reproduction_summary["status"] = "ERROR_NO_JSON"
    print("\n[ERROR] No se generó open_set_nmin_7.json.")

else:
    with open(result_json, "r", encoding="utf-8") as file:
        data = json.load(file)

    osr_cfg = data.get("osr_configuration", {})
    metrics = data.get("metrics", {}).get("binary_detection", {})
    degradation = data.get("metrics", {}).get("known_degradation", {})

    pca_components_real = osr_cfg.get("pca_components")
    pca_variance_actual = osr_cfg.get("pca_variance_actual")
    selected_lambda = osr_cfg.get("selected_lambda_mad")

    ood_auroc = metrics.get("ood_auroc")
    ood_aupr = metrics.get("ood_aupr")
    ood_recall = metrics.get("ood_recall")
    ood_precision = metrics.get("ood_precision")
    ood_f1 = metrics.get("ood_f1")
    known_frr = metrics.get("known_false_rejection_rate")
    binary_mcc = metrics.get("binary_mcc")
    balanced_accuracy = metrics.get("balanced_accuracy")
    mcc_after_shield = degradation.get("mcc_after_shield")

    reproduction_summary.update({
        "status": "OK",
        "pca_components_real": pca_components_real,
        "pca_variance_actual": pca_variance_actual,
        "selected_lambda_mad": selected_lambda,
        "ood_auroc": ood_auroc,
        "ood_aupr": ood_aupr,
        "ood_recall": ood_recall,
        "ood_precision": ood_precision,
        "ood_f1": ood_f1,
        "known_frr": known_frr,
        "binary_mcc": binary_mcc,
        "balanced_accuracy": balanced_accuracy,
        "mcc_after_shield": mcc_after_shield,
    })

    print("\n" + "=" * 80)
    print("RESUMEN DE REPRODUCCIÓN")
    print("=" * 80)
    print("pca_components_real :", pca_components_real)
    print("pca_variance_actual :", pca_variance_actual)
    print("selected_lambda_mad :", selected_lambda)
    print("known_frr           :", known_frr)
    print("ood_recall          :", ood_recall)
    print("ood_precision       :", ood_precision)
    print("ood_f1              :", ood_f1)
    print("ood_auroc           :", ood_auroc)
    print("ood_aupr            :", ood_aupr)
    print("binary_mcc          :", binary_mcc)
    print("balanced_accuracy   :", balanced_accuracy)
    print("mcc_after_shield    :", mcc_after_shield)

    print("\nReferencia esperada aproximada:")
    print("pca_components_real ≈ 257")
    print("ood_auroc ≈ 0.824")
    print("ood_recall ≈ 0.247")

    if (
        pca_components_real is not None
        and abs(int(pca_components_real) - 257) <= 10
        and ood_auroc is not None
        and float(ood_auroc) >= 0.80
    ):
        print("\n✓ Reproducción consistente. Se puede continuar con el barrido de lambdas.")
    else:
        print(
            "\n⚠ La reproducción terminó, pero las métricas no coinciden plenamente "
            "con la referencia. Revisa antes de lanzar el barrido completo."
        )

summary_path = os.path.join(REPRO_DIR, "reproduction_summary.json")

with open(summary_path, "w", encoding="utf-8") as file:
    json.dump(
        reproduction_summary,
        file,
        indent=2,
        ensure_ascii=False,
        sort_keys=True,
    )

print("\nResumen guardado en:", summary_path)

REPRODUCTION_OK = reproduction_summary["status"] == "OK"
print("REPRODUCTION_OK:", REPRODUCTION_OK)

PROJECT_ROOT: /content/osr_workspace/proyecto_ids_vit_colab
RESULTS_DIR: /content/osr_workspace/proyecto_ids_vit_colab/results
REPRO_DIR: /content/osr_workspace/proyecto_ids_vit_colab/results/reproduction_pca_0999999_lambda_3_0_20260618_190133
✓ Rutas críticas localizadas.
✓ Configuración OSR escrita:
  pca_variance_retained: 0.999999
  selected_lambda_mad: 3.0
  lambda_candidates: [1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0, 6.0, 7.0]
✓ Archivos previos limpiados: 0
✓ No se borró mlruns/osr_embedding_cache

Ejecutando:
/usr/bin/python3 -m src.evaluation.evaluate_zero_day --mode prod --n_min 7

[✓] Checkpoint validado: checkpoints/vit_nmin_7_checkpoint.pt
[*] Cargando índice versionado desde data/processed/train_known/dataset_index_v2_train_known_nmin7.pt
[*] Cargando índice versionado desde data/processed/val_known/dataset_index_v2_val_known_nmin7.pt
[*] Cargando índice versionado desde data/processed/test_known/dataset_index_v2_test_known_nmin7.pt
[*] Cargando índice versionado desd

In [24]:
# Persistencia en Drive de índices HDF5 y caché de embeddings OSR
# No copia archivos .tmp incompletos

import os
import subprocess
from datetime import datetime, timezone

PROJECT_ROOT = "/content/osr_workspace/proyecto_ids_vit_colab"
DRIVE_BACKUP_ROOT = "/content/drive/MyDrive/OSR_ViT_Runtime_Cache"

SOURCE_MLRUNS = os.path.join(PROJECT_ROOT, "mlruns")
SOURCE_PROCESSED = os.path.join(PROJECT_ROOT, "data", "processed")

DEST_MLRUNS = os.path.join(DRIVE_BACKUP_ROOT, "mlruns")
DEST_INDEXES = os.path.join(DRIVE_BACKUP_ROOT, "dataset_indexes")
DEST_CONFIGS = os.path.join(DRIVE_BACKUP_ROOT, "configs")

os.makedirs(DEST_MLRUNS, exist_ok=True)
os.makedirs(DEST_INDEXES, exist_ok=True)
os.makedirs(DEST_CONFIGS, exist_ok=True)

print("Backup destino:", DRIVE_BACKUP_ROOT)

# ------------------------------------------------------------
# 1. Verificar Drive
# ------------------------------------------------------------

if not os.path.isdir("/content/drive/MyDrive"):
    raise RuntimeError("Google Drive no parece estar montado en /content/drive/MyDrive")

# ------------------------------------------------------------
# 2. Copiar caché de embeddings, excluyendo temporales
# ------------------------------------------------------------

print("\n[1/3] Copiando mlruns/osr_embedding_cache_nmin_7_* ...")

rsync_mlruns_cmd = [
    "rsync",
    "-avh",
    "--progress",
    "--exclude=*.tmp",
    "--include=osr_embedding_cache_nmin_7_*/",
    "--include=osr_embedding_cache_nmin_7_*/*",
    "--exclude=*",
    SOURCE_MLRUNS + "/",
    DEST_MLRUNS + "/",
]

subprocess.run(rsync_mlruns_cmd, check=True)

# ------------------------------------------------------------
# 3. Copiar índices dataset_index_v2_*_nmin7.pt
# ------------------------------------------------------------

print("\n[2/3] Copiando dataset_index_v2_*_nmin7.pt ...")

splits = ["train_known", "val_known", "test_known", "test_ood"]

for split in splits:
    source_split_dir = os.path.join(SOURCE_PROCESSED, split)
    dest_split_dir = os.path.join(DEST_INDEXES, split)

    os.makedirs(dest_split_dir, exist_ok=True)

    if not os.path.isdir(source_split_dir):
        print(f"[!] No existe: {source_split_dir}")
        continue

    cmd = [
        "find",
        source_split_dir,
        "-maxdepth",
        "1",
        "-type",
        "f",
        "-name",
        "dataset_index_v2_*_nmin7.pt",
        "-exec",
        "cp",
        "-v",
        "{}",
        dest_split_dir,
        ";",
    ]

    subprocess.run(cmd, check=True)

# ------------------------------------------------------------
# 4. Copiar configuración actual para trazabilidad
# ------------------------------------------------------------

print("\n[3/3] Copiando configuración actual ...")

config_files = [
    os.path.join(PROJECT_ROOT, "configs", "global_config.yaml"),
    os.path.join(PROJECT_ROOT, "configs", "dataset_schedule.yaml"),
    os.path.join(PROJECT_ROOT, "configs", "scaler_bounds.json"),
]

for path in config_files:
    if os.path.isfile(path):
        subprocess.run(["cp", "-v", path, DEST_CONFIGS], check=True)

# ------------------------------------------------------------
# 5. Guardar metadata simple
# ------------------------------------------------------------

metadata_path = os.path.join(DRIVE_BACKUP_ROOT, "runtime_cache_backup_info.txt")

with open(metadata_path, "w", encoding="utf-8") as file:
    file.write(f"Backup creado UTC: {datetime.now(timezone.utc).isoformat()}\n")
    file.write(f"PROJECT_ROOT: {PROJECT_ROOT}\n")
    file.write("Contenido:\n")
    file.write("- mlruns/osr_embedding_cache_nmin_7_* sin archivos .tmp\n")
    file.write("- dataset_index_v2_*_nmin7.pt por split\n")
    file.write("- configs relevantes\n")

print("\n✓ Backup de cachés e índices completado.")
print("Metadata:", metadata_path)

print("\nResumen de tamaños en Drive:")
subprocess.run(["du", "-sh", DRIVE_BACKUP_ROOT], check=False)
subprocess.run(["find", DRIVE_BACKUP_ROOT, "-maxdepth", "3", "-type", "f"], check=False)

Backup destino: /content/drive/MyDrive/OSR_ViT_Runtime_Cache

[1/3] Copiando mlruns/osr_embedding_cache_nmin_7_* ...

[2/3] Copiando dataset_index_v2_*_nmin7.pt ...

[3/3] Copiando configuración actual ...

✓ Backup de cachés e índices completado.
Metadata: /content/drive/MyDrive/OSR_ViT_Runtime_Cache/runtime_cache_backup_info.txt

Resumen de tamaños en Drive:


CompletedProcess(args=['find', '/content/drive/MyDrive/OSR_ViT_Runtime_Cache', '-maxdepth', '3', '-type', 'f'], returncode=0)

# 6, 7, 8, 9. Ejecución de barrido

## Sección 6 — Preparar barrido completo de lambdas OSR

In [37]:
# Sección 6 — Preparar barrido completo de lambdas OSR
# PCA fijo 0.999999 + lambdas amplios para análisis completo

import os
import sys
import json
import yaml
import shutil
import subprocess
from datetime import datetime, timezone
from pathlib import Path

# ------------------------------------------------------------
# 1. Rutas base
# ------------------------------------------------------------

PROJECT_ROOT = "/content/osr_workspace/proyecto_ids_vit_colab"
RESULTS_DIR = os.path.join(PROJECT_ROOT, "results")
MLRUNS_DIR = os.path.join(PROJECT_ROOT, "mlruns")
CONFIG_PATH = os.path.join(PROJECT_ROOT, "configs", "global_config.yaml")

os.makedirs(RESULTS_DIR, exist_ok=True)

# ------------------------------------------------------------
# 2. Configuración experimental fija
# ------------------------------------------------------------

N_MIN = 7
PCA_VARIANCE_FIXED = 0.999999

LAMBDAS_TO_RUN = [
    1.0,
    1.25,
    1.5,
    1.75,
    2.0,
]

EVALUATOR_MODULE = "src.evaluation.evaluate_zero_day"

# ------------------------------------------------------------
# 3. Carpeta única para este barrido
# ------------------------------------------------------------

SWEEP_DIR = os.path.join(
    RESULTS_DIR,
    "lambda_sweep_pca_0999999_"
    + datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
)

os.makedirs(SWEEP_DIR, exist_ok=True)

# ------------------------------------------------------------
# 4. Archivos que se limpiarán en cada lambda
# ------------------------------------------------------------
# Importante:
# - Sí se borra osr_profiles_nmin_7.pt para recalcular PCA/Quantile/Mahalanobis.
# - No se borra mlruns/osr_embedding_cache..., porque acelera la ejecución.

GENERATED_PATTERNS = [
    "open_set_nmin_7.json",
    "open_set_nmin_7.txt",
    "open_set_confusion_nmin_7.png",
    "open_set_roc_nmin_7.png",
    "osr_profiles_nmin_7.pt",
]

# ------------------------------------------------------------
# 5. Validaciones previas
# ------------------------------------------------------------

required_paths = {
    "PROJECT_ROOT": PROJECT_ROOT,
    "RESULTS_DIR": RESULTS_DIR,
    "CONFIG_PATH": CONFIG_PATH,
    "evaluate_zero_day.py": os.path.join(
        PROJECT_ROOT,
        "src",
        "evaluation",
        "evaluate_zero_day.py",
    ),
    "checkpoint": os.path.join(
        PROJECT_ROOT,
        "checkpoints",
        "vit_nmin_7_checkpoint.pt",
    ),
    "data/processed": os.path.join(
        PROJECT_ROOT,
        "data",
        "processed",
    ),
}

missing = []

for name, path in required_paths.items():
    if not os.path.exists(path):
        missing.append((name, path))

if missing:
    print("[ERROR] Faltan rutas críticas:")
    for name, path in missing:
        print(f" - {name}: {path}")

    raise FileNotFoundError(
        "No se puede preparar el barrido porque faltan archivos críticos."
    )

print("✓ Rutas críticas localizadas.")

# ------------------------------------------------------------
# 6. Verificar caché de embeddings
# ------------------------------------------------------------

embedding_cache_dirs = []

if os.path.isdir(MLRUNS_DIR):
    embedding_cache_dirs = sorted(
        str(path)
        for path in Path(MLRUNS_DIR).glob("osr_embedding_cache_nmin_7_*")
        if path.is_dir()
    )

if embedding_cache_dirs:
    print("✓ Caché de embeddings detectado:")
    for path in embedding_cache_dirs:
        print(" -", path)
else:
    print(
        "[!] No se detectó caché de embeddings. "
        "La primera corrida puede tardar bastante más."
    )

# ------------------------------------------------------------
# 7. Escribir configuración base del barrido
# ------------------------------------------------------------

with open(CONFIG_PATH, "r", encoding="utf-8") as file:
    config = yaml.safe_load(file)

if "osr_shield" not in config:
    raise KeyError("global_config.yaml no contiene la sección osr_shield")

config["osr_shield"]["pca_variance_retained"] = float(PCA_VARIANCE_FIXED)
config["osr_shield"]["lambda_candidates"] = [float(x) for x in LAMBDAS_TO_RUN]
config["osr_shield"]["selected_lambda_mad"] = None
config["osr_shield"]["max_val_known_frr"] = 0.10
config["osr_shield"]["ridge_epsilon_init"] = 1.0e-5
config["osr_shield"]["quantile_count"] = 10000
config["osr_shield"]["quantile_subsample_size"] = 100000

with open(CONFIG_PATH, "w", encoding="utf-8") as file:
    yaml.safe_dump(
        config,
        file,
        sort_keys=False,
        allow_unicode=True,
    )

shutil.copy2(
    CONFIG_PATH,
    os.path.join(SWEEP_DIR, "global_config_base_for_sweep.yaml"),
)

# ------------------------------------------------------------
# 8. Variables de entorno para ejecución robusta
# ------------------------------------------------------------

RUN_ENV = os.environ.copy()
RUN_ENV["PYTHONUNBUFFERED"] = "1"
RUN_ENV["TOKENIZERS_PARALLELISM"] = "false"

# Ayuda a reducir fragmentación de memoria CUDA en algunos entornos Colab.
RUN_ENV.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

# ------------------------------------------------------------
# 9. Metadata del barrido
# ------------------------------------------------------------

sweep_metadata = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "project_root": PROJECT_ROOT,
    "results_dir": RESULTS_DIR,
    "sweep_dir": SWEEP_DIR,
    "n_min": N_MIN,
    "pca_variance_fixed": PCA_VARIANCE_FIXED,
    "lambdas_to_run": LAMBDAS_TO_RUN,
    "generated_patterns_cleaned_each_run": GENERATED_PATTERNS,
    "embedding_cache_dirs_detected": embedding_cache_dirs,
    "embedding_cache_policy": "preserve mlruns/osr_embedding_cache; only delete osr_profiles_nmin_7.pt per run",
    "python_executable": sys.executable,
    "evaluator_module": EVALUATOR_MODULE,
}

with open(os.path.join(SWEEP_DIR, "sweep_metadata.json"), "w", encoding="utf-8") as file:
    json.dump(
        sweep_metadata,
        file,
        indent=2,
        ensure_ascii=False,
        sort_keys=True,
    )

# ------------------------------------------------------------
# 10. Resumen
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("BARRIDO DE LAMBDAS PREPARADO")
print("=" * 80)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("SWEEP_DIR:", SWEEP_DIR)
print("PCA fijo:", PCA_VARIANCE_FIXED)
print("N_MIN:", N_MIN)
print("Lambdas:", LAMBDAS_TO_RUN)
print("Python:", sys.executable)
print("\nSe limpiará por corrida:")
for item in GENERATED_PATTERNS:
    print(" -", item)

print("\nNo se borrará:")
print(" - mlruns/osr_embedding_cache_nmin_7_*")

print("\n✓ Sección 6 completada. Puedes continuar con las funciones auxiliares del barrido.")

✓ Rutas críticas localizadas.
✓ Caché de embeddings detectado:
 - /content/osr_workspace/proyecto_ids_vit_colab/mlruns/osr_embedding_cache_nmin_7_f835ba3e801a9779

BARRIDO DE LAMBDAS PREPARADO
PROJECT_ROOT: /content/osr_workspace/proyecto_ids_vit_colab
SWEEP_DIR: /content/osr_workspace/proyecto_ids_vit_colab/results/lambda_sweep_pca_0999999_20260618_194721
PCA fijo: 0.999999
N_MIN: 7
Lambdas: [1.0, 1.25, 1.5, 1.75, 2.0]
Python: /usr/bin/python3

Se limpiará por corrida:
 - open_set_nmin_7.json
 - open_set_nmin_7.txt
 - open_set_confusion_nmin_7.png
 - open_set_roc_nmin_7.png
 - osr_profiles_nmin_7.pt

No se borrará:
 - mlruns/osr_embedding_cache_nmin_7_*

✓ Sección 6 completada. Puedes continuar con las funciones auxiliares del barrido.


## Sección 7 — Funciones auxiliares del barrido OSR

In [38]:
# Sección 7 — Funciones auxiliares del barrido OSR
# Objetivo: automatizar cada ejecución por lambda sin borrar el caché de embeddings

import os
import sys
import json
import yaml
import shutil
import subprocess
import gc
from pathlib import Path
from datetime import datetime, timezone

# ------------------------------------------------------------
# 1. Validar que la Sección 6 ya fue ejecutada
# ------------------------------------------------------------

required_variables = [
    "PROJECT_ROOT",
    "RESULTS_DIR",
    "CONFIG_PATH",
    "SWEEP_DIR",
    "PCA_VARIANCE_FIXED",
    "LAMBDAS_TO_RUN",
    "N_MIN",
    "EVALUATOR_MODULE",
]

missing_variables = [
    name for name in required_variables
    if name not in globals()
]

if missing_variables:
    raise RuntimeError(
        "Faltan variables de la Sección 6. Ejecuta primero la Sección 6. "
        f"Variables faltantes: {missing_variables}"
    )

os.makedirs(SWEEP_DIR, exist_ok=True)

# ------------------------------------------------------------
# 2. Archivos generados por evaluate_zero_day.py
# ------------------------------------------------------------

OUTPUT_FILES_TO_CLEAN = [
    f"open_set_nmin_{N_MIN}.json",
    f"open_set_nmin_{N_MIN}.txt",
    f"open_set_confusion_nmin_{N_MIN}.png",
    f"open_set_roc_nmin_{N_MIN}.png",
    f"osr_profiles_nmin_{N_MIN}.pt",
]

OUTPUT_FILES_TO_SAVE = [
    f"open_set_nmin_{N_MIN}.json",
    f"open_set_nmin_{N_MIN}.txt",
    f"open_set_confusion_nmin_{N_MIN}.png",
    f"open_set_roc_nmin_{N_MIN}.png",
]

# Para mayor rapidez no copiamos osr_profiles_nmin_7.pt por defecto.
# Si quieres auditoría completa, cambia esto a True.
COPY_OSR_PROFILE_PT = False

if COPY_OSR_PROFILE_PT:
    OUTPUT_FILES_TO_SAVE.append(f"osr_profiles_nmin_{N_MIN}.pt")

# ------------------------------------------------------------
# 3. Variables de entorno optimizadas para ejecución
# ------------------------------------------------------------

RUN_ENV = os.environ.copy()
RUN_ENV["PYTHONUNBUFFERED"] = "1"
RUN_ENV["TOKENIZERS_PARALLELISM"] = "false"
RUN_ENV.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

# ------------------------------------------------------------
# 4. Función: escribir configuración temporal por lambda
# ------------------------------------------------------------

def set_osr_config_for_lambda(lambda_value):
    lambda_value = float(lambda_value)

    if lambda_value not in [float(x) for x in LAMBDAS_TO_RUN]:
        raise ValueError(
            f"lambda_value={lambda_value} no pertenece a LAMBDAS_TO_RUN={LAMBDAS_TO_RUN}"
        )

    with open(CONFIG_PATH, "r", encoding="utf-8") as file:
        config = yaml.safe_load(file)

    if "osr_shield" not in config:
        raise KeyError("global_config.yaml no contiene la sección osr_shield")

    config["osr_shield"]["pca_variance_retained"] = float(PCA_VARIANCE_FIXED)
    config["osr_shield"]["lambda_candidates"] = [float(x) for x in LAMBDAS_TO_RUN]
    config["osr_shield"]["selected_lambda_mad"] = lambda_value
    config["osr_shield"]["max_val_known_frr"] = 0.10
    config["osr_shield"]["ridge_epsilon_init"] = 1.0e-5
    config["osr_shield"]["quantile_count"] = 10000
    config["osr_shield"]["quantile_subsample_size"] = 100000

    with open(CONFIG_PATH, "w", encoding="utf-8") as file:
        yaml.safe_dump(
            config,
            file,
            sort_keys=False,
            allow_unicode=True,
        )

    return config

# ------------------------------------------------------------
# 5. Función: limpiar resultados previos sin borrar embeddings
# ------------------------------------------------------------

def clean_previous_outputs():
    removed_files = []

    for filename in OUTPUT_FILES_TO_CLEAN:
        path = os.path.join(RESULTS_DIR, filename)

        if os.path.exists(path):
            os.remove(path)
            removed_files.append(path)

    return removed_files

# ------------------------------------------------------------
# 6. Función: ejecutar evaluate_zero_day.py
# ------------------------------------------------------------

def run_zero_day_evaluation():
    cmd = [
        sys.executable,
        "-m",
        EVALUATOR_MODULE,
        "--mode",
        "prod",
        "--n_min",
        str(N_MIN),
    ]

    process = subprocess.Popen(
        cmd,
        cwd=PROJECT_ROOT,
        env=RUN_ENV,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    stdout_lines = []

    for line in process.stdout:
        print(line, end="")
        stdout_lines.append(line)

    returncode = process.wait()
    stdout_text = "".join(stdout_lines)

    return returncode, stdout_text, cmd

# ------------------------------------------------------------
# 7. Función: guardar resultados por lambda
# ------------------------------------------------------------

def save_lambda_outputs(lambda_value, stdout_text, returncode, cmd, config_used, removed_files):
    lambda_tag = str(lambda_value).replace(".", "_")

    lambda_dir = os.path.join(
        SWEEP_DIR,
        f"lambda_{lambda_tag}",
    )

    os.makedirs(lambda_dir, exist_ok=True)

    log_path = os.path.join(lambda_dir, "execution_log.txt")
    config_path_used = os.path.join(lambda_dir, "global_config_used.yaml")
    metadata_path = os.path.join(lambda_dir, "run_metadata.json")

    with open(log_path, "w", encoding="utf-8") as file:
        file.write(stdout_text)

    with open(config_path_used, "w", encoding="utf-8") as file:
        yaml.safe_dump(
            config_used,
            file,
            sort_keys=False,
            allow_unicode=True,
        )

    copied_outputs = []

    for filename in OUTPUT_FILES_TO_SAVE:
        source = os.path.join(RESULTS_DIR, filename)

        if os.path.exists(source):
            target = os.path.join(lambda_dir, filename)
            shutil.copy2(source, target)
            copied_outputs.append(target)

    metadata = {
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "lambda_mad": float(lambda_value),
        "pca_variance_retained": float(PCA_VARIANCE_FIXED),
        "n_min": int(N_MIN),
        "returncode": int(returncode),
        "status": "OK" if returncode == 0 else "ERROR",
        "command": cmd,
        "project_root": PROJECT_ROOT,
        "results_dir": RESULTS_DIR,
        "lambda_dir": lambda_dir,
        "removed_before_run": removed_files,
        "copied_outputs": copied_outputs,
        "copy_osr_profile_pt": COPY_OSR_PROFILE_PT,
        "embedding_cache_policy": "mlruns/osr_embedding_cache no se borra",
    }

    with open(metadata_path, "w", encoding="utf-8") as file:
        json.dump(
            metadata,
            file,
            indent=2,
            ensure_ascii=False,
            sort_keys=True,
        )

    return lambda_dir

# ------------------------------------------------------------
# 8. Función principal: ejecutar una lambda completa
# ------------------------------------------------------------

def run_single_lambda(lambda_value):
    print("\n" + "=" * 90)
    print(f"EJECUTANDO LAMBDA = {lambda_value} | PCA = {PCA_VARIANCE_FIXED}")
    print("=" * 90)

    print("[1/5] Escribiendo configuración temporal...")
    config_used = set_osr_config_for_lambda(lambda_value)

    print("[2/5] Limpiando salidas previas...")
    removed_files = clean_previous_outputs()

    print(f"Archivos limpiados: {len(removed_files)}")
    for path in removed_files:
        print(" -", path)

    print("[3/5] Ejecutando evaluación...")
    returncode, stdout_text, cmd = run_zero_day_evaluation()

    print("\n[4/5] Guardando resultados...")
    lambda_dir = save_lambda_outputs(
        lambda_value=lambda_value,
        stdout_text=stdout_text,
        returncode=returncode,
        cmd=cmd,
        config_used=config_used,
        removed_files=removed_files,
    )

    print("[5/5] Liberando memoria...")
    gc.collect()

    try:
        import torch

        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except Exception:
        pass

    print("Lambda:", lambda_value)
    print("Estado:", "OK" if returncode == 0 else "ERROR")
    print("Carpeta:", lambda_dir)

    return {
        "lambda": float(lambda_value),
        "returncode": int(returncode),
        "status": "OK" if returncode == 0 else "ERROR",
        "lambda_dir": lambda_dir,
    }

print("✓ Sección 7 lista.")
print("Funciones disponibles:")
print(" - run_single_lambda(lambda_value)")
print(" - set_osr_config_for_lambda(lambda_value)")
print(" - clean_previous_outputs()")
print(" - run_zero_day_evaluation()")
print(" - save_lambda_outputs(...)")

✓ Sección 7 lista.
Funciones disponibles:
 - run_single_lambda(lambda_value)
 - set_osr_config_for_lambda(lambda_value)
 - clean_previous_outputs()
 - run_zero_day_evaluation()
 - save_lambda_outputs(...)


## Sección 8 — Ejecutar barrido completo de lambdas OSR

In [39]:
# Sección 8 — Ejecutar barrido completo de lambdas OSR
# Ejecuta una evaluación completa por cada lambda sobre test_known + test_ood

import os
import json
import time
import traceback
from datetime import datetime, timezone

# ------------------------------------------------------------
# 1. Validar que las secciones 6 y 7 ya fueron ejecutadas
# ------------------------------------------------------------

required_objects = [
    "PROJECT_ROOT",
    "RESULTS_DIR",
    "CONFIG_PATH",
    "SWEEP_DIR",
    "PCA_VARIANCE_FIXED",
    "LAMBDAS_TO_RUN",
    "N_MIN",
    "EVALUATOR_MODULE",
    "run_single_lambda",
]

missing_objects = [
    name for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Faltan variables o funciones necesarias. "
        "Ejecuta primero las secciones 6 y 7. "
        f"Faltantes: {missing_objects}"
    )

os.makedirs(SWEEP_DIR, exist_ok=True)

# ------------------------------------------------------------
# 2. Configuración de ejecución del barrido
# ------------------------------------------------------------

FORCE_RERUN = False
STOP_ON_ERROR = False

LAMBDAS_EXECUTION = [float(value) for value in LAMBDAS_TO_RUN]

progress_path = os.path.join(SWEEP_DIR, "sweep_execution_progress.json")
final_status_path = os.path.join(SWEEP_DIR, "sweep_execution_final_status.json")

print("=" * 90)
print("INICIO DEL BARRIDO COMPLETO OSR")
print("=" * 90)
print("SWEEP_DIR:", SWEEP_DIR)
print("PCA fijo:", PCA_VARIANCE_FIXED)
print("N_MIN:", N_MIN)
print("Lambdas a ejecutar:", LAMBDAS_EXECUTION)
print("FORCE_RERUN:", FORCE_RERUN)
print("STOP_ON_ERROR:", STOP_ON_ERROR)
print("=" * 90)

# ------------------------------------------------------------
# 3. Funciones auxiliares locales para control de progreso
# ------------------------------------------------------------

def lambda_to_tag(lambda_value):
    return str(float(lambda_value)).replace(".", "_")


def get_lambda_dir(lambda_value):
    return os.path.join(
        SWEEP_DIR,
        f"lambda_{lambda_to_tag(lambda_value)}",
    )


def get_lambda_json_path(lambda_value):
    return os.path.join(
        get_lambda_dir(lambda_value),
        f"open_set_nmin_{N_MIN}.json",
    )


def get_lambda_metadata_path(lambda_value):
    return os.path.join(
        get_lambda_dir(lambda_value),
        "run_metadata.json",
    )


def existing_lambda_is_complete(lambda_value):
    json_path = get_lambda_json_path(lambda_value)
    metadata_path = get_lambda_metadata_path(lambda_value)

    if not os.path.isfile(json_path) or not os.path.isfile(metadata_path):
        return False

    try:
        with open(metadata_path, "r", encoding="utf-8") as file:
            metadata = json.load(file)

        return metadata.get("status") == "OK"

    except Exception:
        return False


def extract_metrics_from_lambda(lambda_value):
    json_path = get_lambda_json_path(lambda_value)

    if not os.path.isfile(json_path):
        return {}

    try:
        with open(json_path, "r", encoding="utf-8") as file:
            data = json.load(file)

        osr_cfg = data.get("osr_configuration", {})
        binary = data.get("metrics", {}).get("binary_detection", {})
        degradation = data.get("metrics", {}).get("known_degradation", {})

        return {
            "pca_components_real": osr_cfg.get("pca_components"),
            "pca_variance_actual": osr_cfg.get("pca_variance_actual"),
            "selected_lambda_mad": osr_cfg.get("selected_lambda_mad"),
            "val_frr": osr_cfg.get("selected_lambda_val_frr"),
            "known_frr": binary.get("known_false_rejection_rate"),
            "ood_recall": binary.get("ood_recall"),
            "ood_precision": binary.get("ood_precision"),
            "ood_f1": binary.get("ood_f1"),
            "ood_auroc": binary.get("ood_auroc"),
            "ood_aupr": binary.get("ood_aupr"),
            "binary_mcc": binary.get("binary_mcc"),
            "balanced_accuracy": binary.get("balanced_accuracy"),
            "mcc_after_shield": degradation.get("mcc_after_shield"),
            "known_rejected": degradation.get("known_rejected"),
        }

    except Exception as error:
        return {
            "metrics_read_error": f"{type(error).__name__}: {error}",
        }


def save_progress(records, status):
    payload = {
        "updated_at_utc": datetime.now(timezone.utc).isoformat(),
        "status": status,
        "project_root": PROJECT_ROOT,
        "sweep_dir": SWEEP_DIR,
        "pca_variance_fixed": float(PCA_VARIANCE_FIXED),
        "n_min": int(N_MIN),
        "lambdas_requested": LAMBDAS_EXECUTION,
        "force_rerun": FORCE_RERUN,
        "stop_on_error": STOP_ON_ERROR,
        "records": records,
    }

    with open(progress_path, "w", encoding="utf-8") as file:
        json.dump(
            payload,
            file,
            indent=2,
            ensure_ascii=False,
            sort_keys=True,
        )

    return payload

# ------------------------------------------------------------
# 4. Ejecutar barrido lambda por lambda
# ------------------------------------------------------------

sweep_records = []
sweep_start = time.perf_counter()

save_progress(sweep_records, status="RUNNING")

for index, lambda_value in enumerate(LAMBDAS_EXECUTION, start=1):
    lambda_start = time.perf_counter()

    print("\n" + "#" * 90)
    print(f"LAMBDA {index}/{len(LAMBDAS_EXECUTION)}: {lambda_value}")
    print("#" * 90)

    record = {
        "lambda": float(lambda_value),
        "index": index,
        "total_lambdas": len(LAMBDAS_EXECUTION),
        "started_at_utc": datetime.now(timezone.utc).isoformat(),
        "status": "UNKNOWN",
        "lambda_dir": get_lambda_dir(lambda_value),
    }

    try:
        if existing_lambda_is_complete(lambda_value) and not FORCE_RERUN:
            print(f"✓ Lambda {lambda_value} ya tenía resultados completos. Se omite.")

            record["status"] = "SKIPPED_ALREADY_COMPLETE"
            record["returncode"] = 0
            record["metrics"] = extract_metrics_from_lambda(lambda_value)

        else:
            result = run_single_lambda(lambda_value)

            record["status"] = result.get("status", "UNKNOWN")
            record["returncode"] = result.get("returncode")
            record["lambda_dir"] = result.get("lambda_dir", get_lambda_dir(lambda_value))
            record["metrics"] = extract_metrics_from_lambda(lambda_value)

            if record["status"] != "OK":
                print(f"[ERROR] Falló lambda={lambda_value}")

                if STOP_ON_ERROR:
                    record["stop_reason"] = "STOP_ON_ERROR=True"
                    sweep_records.append(record)
                    save_progress(sweep_records, status="STOPPED_ON_ERROR")
                    raise RuntimeError(f"Barrido detenido por error en lambda={lambda_value}")

    except Exception as error:
        record["status"] = "EXCEPTION"
        record["error_type"] = type(error).__name__
        record["error_message"] = str(error)
        record["traceback"] = traceback.format_exc()

        print(f"[ERROR] Excepción en lambda={lambda_value}: {type(error).__name__}: {error}")

        if STOP_ON_ERROR:
            sweep_records.append(record)
            save_progress(sweep_records, status="STOPPED_ON_EXCEPTION")
            raise

    finally:
        lambda_duration = time.perf_counter() - lambda_start

        record["finished_at_utc"] = datetime.now(timezone.utc).isoformat()
        record["duration_seconds"] = round(lambda_duration, 2)
        record["duration_minutes"] = round(lambda_duration / 60.0, 2)

        sweep_records.append(record)
        save_progress(sweep_records, status="RUNNING")

        print(f"Duración lambda {lambda_value}: {record['duration_minutes']} min")
        print("Estado:", record["status"])

# ------------------------------------------------------------
# 5. Guardar estado final del barrido
# ------------------------------------------------------------

total_duration = time.perf_counter() - sweep_start

ok_count = sum(1 for item in sweep_records if item.get("status") == "OK")
skipped_count = sum(1 for item in sweep_records if item.get("status") == "SKIPPED_ALREADY_COMPLETE")
error_count = sum(
    1 for item in sweep_records
    if item.get("status") not in ["OK", "SKIPPED_ALREADY_COMPLETE"]
)

final_payload = {
    "finished_at_utc": datetime.now(timezone.utc).isoformat(),
    "status": "COMPLETED_WITH_ERRORS" if error_count > 0 else "COMPLETED_OK",
    "project_root": PROJECT_ROOT,
    "sweep_dir": SWEEP_DIR,
    "pca_variance_fixed": float(PCA_VARIANCE_FIXED),
    "n_min": int(N_MIN),
    "lambdas_requested": LAMBDAS_EXECUTION,
    "total_lambdas": len(LAMBDAS_EXECUTION),
    "ok_count": ok_count,
    "skipped_count": skipped_count,
    "error_count": error_count,
    "total_duration_seconds": round(total_duration, 2),
    "total_duration_minutes": round(total_duration / 60.0, 2),
    "records": sweep_records,
}

with open(final_status_path, "w", encoding="utf-8") as file:
    json.dump(
        final_payload,
        file,
        indent=2,
        ensure_ascii=False,
        sort_keys=True,
    )

save_progress(sweep_records, status=final_payload["status"])

print("\n" + "=" * 90)
print("BARRIDO COMPLETO FINALIZADO")
print("=" * 90)
print("Estado final:", final_payload["status"])
print("Lambdas OK:", ok_count)
print("Lambdas omitidos:", skipped_count)
print("Lambdas con error:", error_count)
print("Duración total:", final_payload["total_duration_minutes"], "min")
print("SWEEP_DIR:", SWEEP_DIR)
print("Progreso:", progress_path)
print("Estado final:", final_status_path)
print("=" * 90)

SWEEP_EXECUTION_OK = error_count == 0
print("SWEEP_EXECUTION_OK:", SWEEP_EXECUTION_OK)

INICIO DEL BARRIDO COMPLETO OSR
SWEEP_DIR: /content/osr_workspace/proyecto_ids_vit_colab/results/lambda_sweep_pca_0999999_20260618_194721
PCA fijo: 0.999999
N_MIN: 7
Lambdas a ejecutar: [1.0, 1.25, 1.5, 1.75, 2.0]
FORCE_RERUN: False
STOP_ON_ERROR: False

##########################################################################################
LAMBDA 1/5: 1.0
##########################################################################################

EJECUTANDO LAMBDA = 1.0 | PCA = 0.999999
[1/5] Escribiendo configuración temporal...
[2/5] Limpiando salidas previas...
Archivos limpiados: 0
[3/5] Ejecutando evaluación...
[✓] Checkpoint validado: checkpoints/vit_nmin_7_checkpoint.pt
[*] Cargando índice versionado desde data/processed/train_known/dataset_index_v2_train_known_nmin7.pt
[*] Cargando índice versionado desde data/processed/val_known/dataset_index_v2_val_known_nmin7.pt
[*] Cargando índice versionado desde data/processed/test_known/dataset_index_v2_test_known_nmin7.pt
[*] Cargand

## Sección 9 — Tabla resumen comparativa de lambdas OSR

In [40]:
# Sección 9 — Tabla resumen comparativa de lambdas OSR
# Lee cada open_set_nmin_7.json y construye una tabla objetiva para comparar lambdas

import os
import json
import math
import pandas as pd
from datetime import datetime, timezone

# ------------------------------------------------------------
# 1. Validar que las secciones previas existan
# ------------------------------------------------------------

required_variables = [
    "SWEEP_DIR",
    "N_MIN",
    "LAMBDAS_TO_RUN",
]

missing_variables = [
    name for name in required_variables
    if name not in globals()
]

if missing_variables:
    raise RuntimeError(
        "Faltan variables necesarias. Ejecuta primero las secciones 6, 7 y 8. "
        f"Variables faltantes: {missing_variables}"
    )

if not os.path.isdir(SWEEP_DIR):
    raise FileNotFoundError(f"No existe SWEEP_DIR: {SWEEP_DIR}")

print("SWEEP_DIR:", SWEEP_DIR)
print("N_MIN:", N_MIN)
print("Lambdas esperados:", LAMBDAS_TO_RUN)

# ------------------------------------------------------------
# 2. Funciones auxiliares robustas
# ------------------------------------------------------------

def lambda_to_tag(lambda_value):
    return str(float(lambda_value)).replace(".", "_")


def get_nested(dictionary, path, default=None):
    current = dictionary

    for key in path:
        if not isinstance(current, dict) or key not in current:
            return default

        current = current[key]

    return current


def to_float_or_none(value):
    if value is None:
        return None

    try:
        value = float(value)

        if math.isnan(value) or math.isinf(value):
            return None

        return value

    except Exception:
        return None


def to_int_or_none(value):
    if value is None:
        return None

    try:
        value = int(value)
        return value

    except Exception:
        return None


def read_json_safely(path):
    try:
        with open(path, "r", encoding="utf-8") as file:
            return json.load(file), None

    except Exception as error:
        return None, f"{type(error).__name__}: {error}"


# ------------------------------------------------------------
# 3. Leer resultados por lambda
# ------------------------------------------------------------

rows = []

for lambda_value in [float(x) for x in LAMBDAS_TO_RUN]:
    lambda_tag = lambda_to_tag(lambda_value)
    lambda_dir = os.path.join(SWEEP_DIR, f"lambda_{lambda_tag}")

    result_json_path = os.path.join(
        lambda_dir,
        f"open_set_nmin_{N_MIN}.json",
    )

    metadata_path = os.path.join(
        lambda_dir,
        "run_metadata.json",
    )

    row = {
        "lambda": lambda_value,
        "status": "UNKNOWN",
        "pca_components_real": None,
        "val_frr": None,
        "known_frr": None,
        "ood_recall": None,
        "ood_precision": None,
        "ood_f1": None,
        "ood_auroc": None,
        "ood_aupr": None,
        "binary_mcc": None,
        "balanced_accuracy": None,
        "mcc_after_shield": None,
        "known_rejected": None,
        "result_json_path": result_json_path,
        "lambda_dir": lambda_dir,
    }

    metadata, metadata_error = read_json_safely(metadata_path)

    if metadata is not None:
        row["status"] = metadata.get("status", "UNKNOWN")
        row["duration_minutes"] = metadata.get("duration_minutes")
    else:
        row["duration_minutes"] = None

    if not os.path.isfile(result_json_path):
        row["status"] = "MISSING_JSON"
        row["read_error"] = "No existe open_set_nmin JSON"
        rows.append(row)
        continue

    data, read_error = read_json_safely(result_json_path)

    if data is None:
        row["status"] = "JSON_READ_ERROR"
        row["read_error"] = read_error
        rows.append(row)
        continue

    osr_cfg = data.get("osr_configuration", {})
    binary = get_nested(data, ["metrics", "binary_detection"], default={})
    degradation = get_nested(data, ["metrics", "known_degradation"], default={})

    row.update({
        "status": row["status"] if row["status"] != "UNKNOWN" else "OK",
        "pca_components_real": to_int_or_none(osr_cfg.get("pca_components")),
        "pca_variance_actual": to_float_or_none(osr_cfg.get("pca_variance_actual")),
        "selected_lambda_mad": to_float_or_none(osr_cfg.get("selected_lambda_mad")),
        "val_frr": to_float_or_none(osr_cfg.get("selected_lambda_val_frr")),
        "known_frr": to_float_or_none(binary.get("known_false_rejection_rate")),
        "ood_recall": to_float_or_none(binary.get("ood_recall")),
        "ood_precision": to_float_or_none(binary.get("ood_precision")),
        "ood_f1": to_float_or_none(binary.get("ood_f1")),
        "ood_auroc": to_float_or_none(binary.get("ood_auroc")),
        "ood_aupr": to_float_or_none(binary.get("ood_aupr")),
        "binary_mcc": to_float_or_none(binary.get("binary_mcc")),
        "balanced_accuracy": to_float_or_none(binary.get("balanced_accuracy")),
        "mcc_after_shield": to_float_or_none(degradation.get("mcc_after_shield")),
        "known_rejected": to_int_or_none(degradation.get("known_rejected")),
        "read_error": None,
    })

    rows.append(row)

# ------------------------------------------------------------
# 4. Construir DataFrame comparativo
# ------------------------------------------------------------

summary_df = pd.DataFrame(rows)

ordered_columns = [
    "lambda",
    "status",
    "pca_components_real",
    "pca_variance_actual",
    "selected_lambda_mad",
    "val_frr",
    "known_frr",
    "ood_recall",
    "ood_precision",
    "ood_f1",
    "ood_auroc",
    "ood_aupr",
    "binary_mcc",
    "balanced_accuracy",
    "mcc_after_shield",
    "known_rejected",
    "duration_minutes",
    "read_error",
    "result_json_path",
    "lambda_dir",
]

for column in ordered_columns:
    if column not in summary_df.columns:
        summary_df[column] = None

summary_df = summary_df[ordered_columns].sort_values("lambda").reset_index(drop=True)

# ------------------------------------------------------------
# 5. Guardar tabla resumen
# ------------------------------------------------------------

summary_csv_path = os.path.join(
    SWEEP_DIR,
    "lambda_sweep_summary.csv",
)

summary_json_path = os.path.join(
    SWEEP_DIR,
    "lambda_sweep_summary.json",
)

summary_markdown_path = os.path.join(
    SWEEP_DIR,
    "lambda_sweep_summary.md",
)

summary_df.to_csv(summary_csv_path, index=False)

with open(summary_json_path, "w", encoding="utf-8") as file:
    json.dump(
        {
            "created_at_utc": datetime.now(timezone.utc).isoformat(),
            "sweep_dir": SWEEP_DIR,
            "n_min": int(N_MIN),
            "records": summary_df.to_dict(orient="records"),
        },
        file,
        indent=2,
        ensure_ascii=False,
        sort_keys=True,
    )

with open(summary_markdown_path, "w", encoding="utf-8") as file:
    file.write(summary_df.to_markdown(index=False))

# ------------------------------------------------------------
# 6. Mostrar tabla y ranking preliminar
# ------------------------------------------------------------

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

print("\n" + "=" * 90)
print("TABLA RESUMEN COMPARATIVA")
print("=" * 90)

display(summary_df)

valid_df = summary_df[
    summary_df["status"].isin(["OK", "SKIPPED_ALREADY_COMPLETE"])
].copy()

if valid_df.empty:
    print("\n[!] No hay resultados válidos para comparar.")
else:
    ranking_df = valid_df.sort_values(
        by=[
            "binary_mcc",
            "balanced_accuracy",
            "ood_auroc",
            "ood_recall",
            "known_frr",
        ],
        ascending=[
            False,
            False,
            False,
            False,
            True,
        ],
    ).reset_index(drop=True)

    print("\n" + "=" * 90)
    print("RANKING PRELIMINAR")
    print("=" * 90)

    display(
        ranking_df[
            [
                "lambda",
                "known_frr",
                "ood_recall",
                "ood_precision",
                "ood_f1",
                "ood_auroc",
                "ood_aupr",
                "binary_mcc",
                "balanced_accuracy",
                "mcc_after_shield",
                "known_rejected",
            ]
        ]
    )

    best_row = ranking_df.iloc[0]

    print("\nMejor candidato preliminar según ranking:")
    print("lambda:", best_row["lambda"])
    print("binary_mcc:", best_row["binary_mcc"])
    print("balanced_accuracy:", best_row["balanced_accuracy"])
    print("ood_auroc:", best_row["ood_auroc"])
    print("ood_recall:", best_row["ood_recall"])
    print("known_frr:", best_row["known_frr"])

print("\nArchivos guardados:")
print("CSV:", summary_csv_path)
print("JSON:", summary_json_path)
print("Markdown:", summary_markdown_path)

LAMBDA_SWEEP_SUMMARY_DF = summary_df
LAMBDA_SWEEP_RANKING_DF = ranking_df if "ranking_df" in locals() else None

print("\n✓ Sección 9 completada.")

SWEEP_DIR: /content/osr_workspace/proyecto_ids_vit_colab/results/lambda_sweep_pca_0999999_20260618_194721
N_MIN: 7
Lambdas esperados: [1.0, 1.25, 1.5, 1.75, 2.0]

TABLA RESUMEN COMPARATIVA


,lambda,status,pca_components_real,pca_variance_actual,selected_lambda_mad,val_frr,known_frr,ood_recall,ood_precision,ood_f1,ood_auroc,ood_aupr,binary_mcc,balanced_accuracy,mcc_after_shield,known_rejected,duration_minutes,read_error,result_json_path,lambda_dir
0,1.00,OK,257,0.999999,1.00,0.293523,0.294189,0.990467,0.826588,0.901137,0.832281,0.751193,0.749980,0.848139,0.530922,19966,None,None,/content/osr_workspace/proyecto_ids_vit_colab/...,/content/osr_workspace/proyecto_ids_vit_colab/...
1,1.25,OK,257,0.999999,1.25,0.261124,0.261095,0.961691,0.839092,0.896218,0.831159,0.750191,0.734816,0.850298,0.563138,17720,None,None,/content/osr_workspace/proyecto_ids_vit_colab/...,/content/osr_workspace/proyecto_ids_vit_colab/...
2,1.50,OK,257,0.999999,1.50,0.234400,0.234735,0.900631,0.844529,0.871678,0.830071,0.749222,0.677467,0.832948,0.591179,15931,None,None,/content/osr_workspace/proyecto_ids_vit_colab/...,/content/osr_workspace/proyecto_ids_vit_colab/...
3,1.75,OK,257,0.999999,1.75,0.211299,0.212471,0.804009,0.842703,0.822902,0.829016,0.748284,0.586831,0.795769,0.616397,14420,None,None,/content/osr_workspace/proyecto_ids_vit_colab/...,/content/osr_workspace/proyecto_ids_vit_colab/...
4,2.00,OK,257,0.999999,2.00,0.192876,0.193287,0.683554,0.833524,0.751126,0.827996,0.747377,0.483315,0.745134,0.639854,13118,None,None,/content/osr_workspace/proyecto_ids_vit_colab/...,/content/osr_workspace/proyecto_ids_vit_colab/...



RANKING PRELIMINAR


,lambda,known_frr,ood_recall,ood_precision,ood_f1,ood_auroc,ood_aupr,binary_mcc,balanced_accuracy,mcc_after_shield,known_rejected
0,1.00,0.294189,0.990467,0.826588,0.901137,0.832281,0.751193,0.749980,0.848139,0.530922,19966
1,1.25,0.261095,0.961691,0.839092,0.896218,0.831159,0.750191,0.734816,0.850298,0.563138,17720
2,1.50,0.234735,0.900631,0.844529,0.871678,0.830071,0.749222,0.677467,0.832948,0.591179,15931
3,1.75,0.212471,0.804009,0.842703,0.822902,0.829016,0.748284,0.586831,0.795769,0.616397,14420
4,2.00,0.193287,0.683554,0.833524,0.751126,0.827996,0.747377,0.483315,0.745134,0.639854,13118



Mejor candidato preliminar según ranking:
lambda: 1.0
binary_mcc: 0.7499801896812551
balanced_accuracy: 0.8481390770736448
ood_auroc: 0.8322810490093302
ood_recall: 0.9904668734258893
known_frr: 0.29418871927859963

Archivos guardados:
CSV: /content/osr_workspace/proyecto_ids_vit_colab/results/lambda_sweep_pca_0999999_20260618_194721/lambda_sweep_summary.csv
JSON: /content/osr_workspace/proyecto_ids_vit_colab/results/lambda_sweep_pca_0999999_20260618_194721/lambda_sweep_summary.json
Markdown: /content/osr_workspace/proyecto_ids_vit_colab/results/lambda_sweep_pca_0999999_20260618_194721/lambda_sweep_summary.md

✓ Sección 9 completada.


# 10. Selección manual del lambda final

In [42]:
# Sección 10 — Selección manual del lambda final

FINAL_LAMBDA_MAD = 1.25  # Cambiar por el lambda elegido tras revisar la tabla

FINAL_SELECTION_REASON = (
    "Se selecciona este lambda porque ofrece el mejor compromiso entre "
    "detección OOD, MCC/balanced accuracy y preservación de muestras conocidas."
)

print("Lambda final seleccionado:", FINAL_LAMBDA_MAD)
print("Justificación:", FINAL_SELECTION_REASON)

Lambda final seleccionado: 1.25
Justificación: Se selecciona este lambda porque ofrece el mejor compromiso entre detección OOD, MCC/balanced accuracy y preservación de muestras conocidas.


# 11. Ejecución final bloqueada OSR

In [43]:
# Sección 11 — Ejecución final bloqueada OSR
# Objetivo: correr una última evaluación limpia con el lambda seleccionado
# Salida oficial:
# - open_set_nmin_7.json
# - open_set_nmin_7.txt
# - open_set_confusion_nmin_7.png
# - open_set_roc_nmin_7.png
# - global_config_used.yaml

import os
import sys
import json
import yaml
import shutil
import subprocess
import gc
from datetime import datetime, timezone

# ------------------------------------------------------------
# 1. Validar lambda final seleccionado en la Sección 10
# ------------------------------------------------------------

if "FINAL_LAMBDA_MAD" not in globals():
    raise RuntimeError(
        "No existe FINAL_LAMBDA_MAD. Ejecuta primero la Sección 10 "
        "y define manualmente el lambda final."
    )

FINAL_LAMBDA_MAD = float(FINAL_LAMBDA_MAD)

if FINAL_LAMBDA_MAD <= 0:
    raise ValueError("FINAL_LAMBDA_MAD debe ser un número positivo.")

# ------------------------------------------------------------
# 2. Rutas y configuración base
# ------------------------------------------------------------

PROJECT_ROOT = globals().get(
    "PROJECT_ROOT",
    "/content/osr_workspace/proyecto_ids_vit_colab",
)

RESULTS_DIR = globals().get(
    "RESULTS_DIR",
    os.path.join(PROJECT_ROOT, "results"),
)

CONFIG_PATH = globals().get(
    "CONFIG_PATH",
    os.path.join(PROJECT_ROOT, "configs", "global_config.yaml"),
)

N_MIN = int(globals().get("N_MIN", 7))

PCA_VARIANCE_FIXED = float(
    globals().get("PCA_VARIANCE_FIXED", 0.999999)
)

EVALUATOR_MODULE = globals().get(
    "EVALUATOR_MODULE",
    "src.evaluation.evaluate_zero_day",
)

os.makedirs(RESULTS_DIR, exist_ok=True)

FINAL_OFFICIAL_DIR = os.path.join(
    RESULTS_DIR,
    f"final_official_nmin_{N_MIN}_lambda_{str(FINAL_LAMBDA_MAD).replace('.', '_')}",
)

# Si ya existe una ejecución final previa, se respalda antes de sobrescribir.
if os.path.isdir(FINAL_OFFICIAL_DIR):
    backup_dir = (
        FINAL_OFFICIAL_DIR
        + "_backup_"
        + datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
    )
    shutil.move(FINAL_OFFICIAL_DIR, backup_dir)
    print("Backup de ejecución final previa:", backup_dir)

os.makedirs(FINAL_OFFICIAL_DIR, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RESULTS_DIR:", RESULTS_DIR)
print("FINAL_OFFICIAL_DIR:", FINAL_OFFICIAL_DIR)
print("PCA fijo:", PCA_VARIANCE_FIXED)
print("Lambda final:", FINAL_LAMBDA_MAD)
print("N_MIN:", N_MIN)

# ------------------------------------------------------------
# 3. Validaciones mínimas de archivos críticos
# ------------------------------------------------------------

required_paths = {
    "global_config.yaml": CONFIG_PATH,
    "evaluate_zero_day.py": os.path.join(
        PROJECT_ROOT,
        "src",
        "evaluation",
        "evaluate_zero_day.py",
    ),
    "checkpoint": os.path.join(
        PROJECT_ROOT,
        "checkpoints",
        f"vit_nmin_{N_MIN}_checkpoint.pt",
    ),
    "data/processed": os.path.join(
        PROJECT_ROOT,
        "data",
        "processed",
    ),
}

missing_paths = []

for name, path in required_paths.items():
    if not os.path.exists(path):
        missing_paths.append((name, path))

if missing_paths:
    print("[ERROR] Faltan rutas críticas:")
    for name, path in missing_paths:
        print(f" - {name}: {path}")

    raise FileNotFoundError(
        "No se puede ejecutar la evaluación final porque faltan archivos críticos."
    )

print("✓ Rutas críticas localizadas.")

# ------------------------------------------------------------
# 4. Escribir configuración final bloqueada
# ------------------------------------------------------------

with open(CONFIG_PATH, "r", encoding="utf-8") as file:
    config = yaml.safe_load(file)

if "osr_shield" not in config:
    raise KeyError("global_config.yaml no contiene la sección osr_shield")

existing_candidates = config["osr_shield"].get("lambda_candidates", [])
existing_candidates = [float(x) for x in existing_candidates]

lambda_candidates_final = sorted(
    set(existing_candidates + [FINAL_LAMBDA_MAD])
)

config["osr_shield"]["pca_variance_retained"] = PCA_VARIANCE_FIXED
config["osr_shield"]["lambda_candidates"] = lambda_candidates_final
config["osr_shield"]["selected_lambda_mad"] = FINAL_LAMBDA_MAD
config["osr_shield"]["max_val_known_frr"] = 0.10
config["osr_shield"]["ridge_epsilon_init"] = 1.0e-5
config["osr_shield"]["quantile_count"] = 10000
config["osr_shield"]["quantile_subsample_size"] = 100000

with open(CONFIG_PATH, "w", encoding="utf-8") as file:
    yaml.safe_dump(
        config,
        file,
        sort_keys=False,
        allow_unicode=True,
    )

global_config_used_path = os.path.join(
    FINAL_OFFICIAL_DIR,
    "global_config_used.yaml",
)

shutil.copy2(CONFIG_PATH, global_config_used_path)

print("✓ Configuración final escrita:")
print("  pca_variance_retained:", config["osr_shield"]["pca_variance_retained"])
print("  selected_lambda_mad:", config["osr_shield"]["selected_lambda_mad"])
print("  lambda_candidates:", config["osr_shield"]["lambda_candidates"])

# ------------------------------------------------------------
# 5. Limpiar salidas previas sin borrar caché de embeddings
# ------------------------------------------------------------

outputs_to_clean = [
    f"open_set_nmin_{N_MIN}.json",
    f"open_set_nmin_{N_MIN}.txt",
    f"open_set_confusion_nmin_{N_MIN}.png",
    f"open_set_roc_nmin_{N_MIN}.png",
    f"osr_profiles_nmin_{N_MIN}.pt",
]

removed_files = []

for filename in outputs_to_clean:
    path = os.path.join(RESULTS_DIR, filename)

    if os.path.exists(path):
        os.remove(path)
        removed_files.append(path)

print(f"✓ Archivos previos limpiados: {len(removed_files)}")
for path in removed_files:
    print(" -", path)

print("✓ No se borró mlruns/osr_embedding_cache")

# ------------------------------------------------------------
# 6. Ejecutar evaluación final bloqueada
# ------------------------------------------------------------

run_env = os.environ.copy()
run_env["PYTHONUNBUFFERED"] = "1"
run_env["TOKENIZERS_PARALLELISM"] = "false"
run_env.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

cmd = [
    sys.executable,
    "-m",
    EVALUATOR_MODULE,
    "--mode",
    "prod",
    "--n_min",
    str(N_MIN),
]

print("\nEjecutando evaluación final:")
print(" ".join(cmd))
print()

process = subprocess.Popen(
    cmd,
    cwd=PROJECT_ROOT,
    env=run_env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

stdout_lines = []

for line in process.stdout:
    print(line, end="")
    stdout_lines.append(line)

returncode = process.wait()
stdout_text = "".join(stdout_lines)

execution_log_path = os.path.join(
    FINAL_OFFICIAL_DIR,
    "execution_log.txt",
)

with open(execution_log_path, "w", encoding="utf-8") as file:
    file.write(stdout_text)

print("\nReturn code:", returncode)
print("Log guardado en:", execution_log_path)

# ------------------------------------------------------------
# 7. Copiar artefactos oficiales
# ------------------------------------------------------------

official_outputs = [
    f"open_set_nmin_{N_MIN}.json",
    f"open_set_nmin_{N_MIN}.txt",
    f"open_set_confusion_nmin_{N_MIN}.png",
    f"open_set_roc_nmin_{N_MIN}.png",
]

copied_outputs = []

for filename in official_outputs:
    source = os.path.join(RESULTS_DIR, filename)
    target = os.path.join(FINAL_OFFICIAL_DIR, filename)

    if os.path.exists(source):
        shutil.copy2(source, target)
        copied_outputs.append(target)

missing_outputs = [
    filename
    for filename in official_outputs
    if not os.path.exists(os.path.join(FINAL_OFFICIAL_DIR, filename))
]

print(f"✓ Artefactos oficiales copiados: {len(copied_outputs)}")
for path in copied_outputs:
    print(" -", path)

if missing_outputs:
    print("[ADVERTENCIA] Faltan artefactos oficiales:")
    for filename in missing_outputs:
        print(" -", filename)

# ------------------------------------------------------------
# 8. Leer métricas finales y guardar metadata
# ------------------------------------------------------------

final_json_path = os.path.join(
    FINAL_OFFICIAL_DIR,
    f"open_set_nmin_{N_MIN}.json",
)

final_metadata = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "status": "OK" if returncode == 0 and os.path.isfile(final_json_path) else "ERROR",
    "returncode": int(returncode),
    "project_root": PROJECT_ROOT,
    "results_dir": RESULTS_DIR,
    "final_official_dir": FINAL_OFFICIAL_DIR,
    "n_min": int(N_MIN),
    "pca_variance_retained": float(PCA_VARIANCE_FIXED),
    "final_lambda_mad": float(FINAL_LAMBDA_MAD),
    "command": cmd,
    "global_config_used": global_config_used_path,
    "execution_log": execution_log_path,
    "copied_outputs": copied_outputs,
    "missing_outputs": missing_outputs,
    "embedding_cache_policy": "mlruns/osr_embedding_cache no se borra",
}

if os.path.isfile(final_json_path):
    with open(final_json_path, "r", encoding="utf-8") as file:
        final_data = json.load(file)

    osr_cfg = final_data.get("osr_configuration", {})
    binary = final_data.get("metrics", {}).get("binary_detection", {})
    degradation = final_data.get("metrics", {}).get("known_degradation", {})

    final_metadata["metrics"] = {
        "pca_components_real": osr_cfg.get("pca_components"),
        "pca_variance_actual": osr_cfg.get("pca_variance_actual"),
        "selected_lambda_mad": osr_cfg.get("selected_lambda_mad"),
        "val_frr": osr_cfg.get("selected_lambda_val_frr"),
        "known_frr": binary.get("known_false_rejection_rate"),
        "ood_recall": binary.get("ood_recall"),
        "ood_precision": binary.get("ood_precision"),
        "ood_f1": binary.get("ood_f1"),
        "ood_auroc": binary.get("ood_auroc"),
        "ood_aupr": binary.get("ood_aupr"),
        "binary_mcc": binary.get("binary_mcc"),
        "balanced_accuracy": binary.get("balanced_accuracy"),
        "mcc_after_shield": degradation.get("mcc_after_shield"),
        "known_rejected": degradation.get("known_rejected"),
    }

metadata_path = os.path.join(
    FINAL_OFFICIAL_DIR,
    "final_official_metadata.json",
)

with open(metadata_path, "w", encoding="utf-8") as file:
    json.dump(
        final_metadata,
        file,
        indent=2,
        ensure_ascii=False,
        sort_keys=True,
    )

# ------------------------------------------------------------
# 9. Liberar memoria y mostrar resumen final
# ------------------------------------------------------------

gc.collect()

try:
    import torch

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
except Exception:
    pass

print("\n" + "=" * 90)
print("EJECUCIÓN FINAL BLOQUEADA TERMINADA")
print("=" * 90)
print("Estado:", final_metadata["status"])
print("Lambda final:", FINAL_LAMBDA_MAD)
print("PCA fijo:", PCA_VARIANCE_FIXED)
print("Carpeta oficial:", FINAL_OFFICIAL_DIR)
print("Metadata:", metadata_path)

if "metrics" in final_metadata:
    print("\nMétricas finales:")
    for key, value in final_metadata["metrics"].items():
        print(f" - {key}: {value}")

print("=" * 90)

FINAL_OFFICIAL_OK = final_metadata["status"] == "OK"
print("FINAL_OFFICIAL_OK:", FINAL_OFFICIAL_OK)

PROJECT_ROOT: /content/osr_workspace/proyecto_ids_vit_colab
RESULTS_DIR: /content/osr_workspace/proyecto_ids_vit_colab/results
FINAL_OFFICIAL_DIR: /content/osr_workspace/proyecto_ids_vit_colab/results/final_official_nmin_7_lambda_1_25
PCA fijo: 0.999999
Lambda final: 1.25
N_MIN: 7
✓ Rutas críticas localizadas.
✓ Configuración final escrita:
  pca_variance_retained: 0.999999
  selected_lambda_mad: 1.25
  lambda_candidates: [1.0, 1.25, 1.5, 1.75, 2.0]
✓ Archivos previos limpiados: 5
 - /content/osr_workspace/proyecto_ids_vit_colab/results/open_set_nmin_7.json
 - /content/osr_workspace/proyecto_ids_vit_colab/results/open_set_nmin_7.txt
 - /content/osr_workspace/proyecto_ids_vit_colab/results/open_set_confusion_nmin_7.png
 - /content/osr_workspace/proyecto_ids_vit_colab/results/open_set_roc_nmin_7.png
 - /content/osr_workspace/proyecto_ids_vit_colab/results/osr_profiles_nmin_7.pt
✓ No se borró mlruns/osr_embedding_cache

Ejecutando evaluación final:
/usr/bin/python3 -m src.evaluation.evalu

# 12. Persistencia

In [44]:
import os
import json
import glob
import shutil
import hashlib
from datetime import datetime

PROJECT_ROOT = "/content/osr_workspace/proyecto_ids_vit_colab"
RESULTS_DIR = os.path.join(PROJECT_ROOT, "results")
CONFIGS_DIR = os.path.join(PROJECT_ROOT, "configs")
SRC_DIR = os.path.join(PROJECT_ROOT, "src")
MLRUNS_DIR = os.path.join(PROJECT_ROOT, "mlruns")
CHECKPOINTS_DIR = os.path.join(PROJECT_ROOT, "checkpoints")

DRIVE_ROOT = "/content/drive/MyDrive/OSR_ViT_Backup_FULL"
DRIVE_ARCHIVE_ROOT = os.path.join(DRIVE_ROOT, "osr_experiment_archives")

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
MASTER_BACKUP_DIR = os.path.join(
    DRIVE_ARCHIVE_ROOT,
    f"osr_vit_full_experiment_lambda_1_25_pca_0999999_{timestamp}"
)

os.makedirs(MASTER_BACKUP_DIR, exist_ok=False)

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def copy_file(src, dst_dir):
    if not os.path.isfile(src):
        return None
    os.makedirs(dst_dir, exist_ok=True)
    dst = os.path.join(dst_dir, os.path.basename(src))
    shutil.copy2(src, dst)
    return dst

def copy_dir(src, dst_parent):
    if not os.path.isdir(src):
        return None
    os.makedirs(dst_parent, exist_ok=True)
    dst = os.path.join(dst_parent, os.path.basename(src))
    if os.path.exists(dst):
        raise RuntimeError(f"No se sobrescribe: {dst}")
    shutil.copytree(src, dst)
    return dst

copied_files = []
copied_dirs = []

# 1. Barridos completos generados
for sweep_dir in sorted(glob.glob(os.path.join(RESULTS_DIR, "lambda_sweep_pca_0999999_*"))):
    dst = copy_dir(sweep_dir, os.path.join(MASTER_BACKUP_DIR, "lambda_sweeps"))
    if dst:
        copied_dirs.append(dst)

# 2. Resultados finales oficiales
final_files = [
    "open_set_nmin_7.json",
    "open_set_nmin_7.txt",
    "open_set_roc_nmin_7.png",
    "open_set_confusion_nmin_7.png",
    "osr_profiles_nmin_7.pt",
    "osr_lambda_sensitivity_nmin_7.csv",
    "osr_lambda_sensitivity_nmin_7.json",
    "osr_lambda_sensitivity_nmin_7.txt",
]

for name in final_files:
    dst = copy_file(
        os.path.join(RESULTS_DIR, name),
        os.path.join(MASTER_BACKUP_DIR, "final_selected_lambda_results")
    )
    if dst:
        copied_files.append(dst)

# 3. Configuración exacta usada
for name in ["global_config.yaml", "dataset_schedule.yaml", "scaler_bounds.json"]:
    dst = copy_file(
        os.path.join(CONFIGS_DIR, name),
        os.path.join(MASTER_BACKUP_DIR, "configs")
    )
    if dst:
        copied_files.append(dst)

# 4. Código crítico usado
critical_code_files = [
    os.path.join(SRC_DIR, "evaluation", "evaluate_zero_day.py"),
    os.path.join(SRC_DIR, "osr_module", "mahalanobis.py"),
    os.path.join(SRC_DIR, "models", "vit_ablation.py"),
    os.path.join(SRC_DIR, "utils", "config_manager.py"),
]

for path in critical_code_files:
    dst = copy_file(path, os.path.join(MASTER_BACKUP_DIR, "src_snapshot"))
    if dst:
        copied_files.append(dst)

# 5. Checkpoint del modelo
for path in sorted(glob.glob(os.path.join(CHECKPOINTS_DIR, "*.pt"))):
    dst = copy_file(path, os.path.join(MASTER_BACKUP_DIR, "checkpoints"))
    if dst:
        copied_files.append(dst)

# 6. Logs
for path in sorted(glob.glob(os.path.join(MLRUNS_DIR, "*.log"))):
    dst = copy_file(path, os.path.join(MASTER_BACKUP_DIR, "logs"))
    if dst:
        copied_files.append(dst)

# 7. Caché de embeddings reutilizable
# Importante: esto acelera futuras ejecuciones.
for cache_dir in sorted(glob.glob(os.path.join(MLRUNS_DIR, "osr_embedding_cache_nmin_7_*"))):
    dst = copy_dir(cache_dir, os.path.join(MASTER_BACKUP_DIR, "embedding_cache"))
    if dst:
        copied_dirs.append(dst)

# 8. No guardar temporales osr_transformed_nmin_7_* porque son recreables
transient_dirs = sorted(glob.glob(os.path.join(MLRUNS_DIR, "osr_transformed_nmin_7_*")))

# 9. Manifest general
manifest = {
    "created_at": timestamp,
    "project_root": PROJECT_ROOT,
    "master_backup_dir": MASTER_BACKUP_DIR,
    "final_lambda_mad": 1.25,
    "pca_variance_retained": 0.999999,
    "n_min": 7,
    "included": {
        "lambda_sweeps": True,
        "final_results": True,
        "configs": True,
        "critical_source_code": True,
        "checkpoints": True,
        "logs": True,
        "embedding_cache": True,
    },
    "excluded_recreable_temporaries": transient_dirs,
    "copied_dirs": copied_dirs,
    "copied_files": [
        {
            "path": path,
            "size_bytes": os.path.getsize(path),
            "sha256": sha256_file(path),
        }
        for path in copied_files
        if os.path.isfile(path)
    ],
}

manifest_path = os.path.join(MASTER_BACKUP_DIR, "master_backup_manifest.json")

with open(manifest_path, "w", encoding="utf-8") as file:
    json.dump(manifest, file, indent=2, ensure_ascii=False)

print("✓ Backup maestro completo guardado en Drive:")
print(MASTER_BACKUP_DIR)
print("\nCarpetas copiadas:", len(copied_dirs))
print("Archivos individuales copiados:", len(copied_files))
print("Manifest:", manifest_path)

✓ Backup maestro completo guardado en Drive:
/content/drive/MyDrive/OSR_ViT_Backup_FULL/osr_experiment_archives/osr_vit_full_experiment_lambda_1_25_pca_0999999_20260618_200726

Carpetas copiadas: 3
Archivos individuales copiados: 17
Manifest: /content/drive/MyDrive/OSR_ViT_Backup_FULL/osr_experiment_archives/osr_vit_full_experiment_lambda_1_25_pca_0999999_20260618_200726/master_backup_manifest.json


In [45]:
import os

BACKUP_DIR = "/content/drive/MyDrive/OSR_ViT_Backup_FULL/osr_experiment_archives/osr_vit_full_experiment_lambda_1_25_pca_0999999_20260618_200726"

for root, dirs, files in os.walk(BACKUP_DIR):
    level = root.replace(BACKUP_DIR, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    for file in files:
        print(f"{indent}  {file}")

osr_vit_full_experiment_lambda_1_25_pca_0999999_20260618_200726/
  master_backup_manifest.json
  lambda_sweeps/
    lambda_sweep_pca_0999999_20260618_191525/
      sweep_metadata.json
      sweep_execution_progress.json
      lambda_sweep_summary.csv
      lambda_sweep_summary.json
      global_config_base_for_sweep.yaml
      lambda_sweep_summary.md
      sweep_execution_final_status.json
      lambda_1_0/
        open_set_confusion_nmin_7.png
        open_set_roc_nmin_7.png
        run_metadata.json
        open_set_nmin_7.json
        global_config_used.yaml
        execution_log.txt
        open_set_nmin_7.txt
      lambda_7_0/
        open_set_confusion_nmin_7.png
        open_set_roc_nmin_7.png
        run_metadata.json
        open_set_nmin_7.json
        global_config_used.yaml
        execution_log.txt
        open_set_nmin_7.txt
      lambda_2_0/
        open_set_confusion_nmin_7.png
        open_set_roc_nmin_7.png
        run_metadata.json
        open_set_nmin_7.json
      